# Standalone DIMSIM geometry analysis

All canonical implementation code is embedded as ordinary cells. No project checkout is required.


In [ ]:
BUILD_METADATA = {'schema_version': 1, 'baseline_git_sha': '1f77df1526f79a9b67fd092adad79dbbf24b8d79', 'source_sha256': {'pinyin.py': '6d080de3613cd219edd5553512e17e4fe596a7bb881d59c0f6acd654bb85f1d3', 'objectives.py': '2cdb95c3044103e6138d28038d696e43b200ea7b0e94c3131cac9a3fd3c0df47', 'artifacts.py': '9447013a45a347b7df6735cde9dc7c40f0931d53792165999c482c384321d64d', 'data_contract.py': '5c526086867ed477286c1f369ca78e1ccd44fa52ed8453846d87a664f2971fa3', 'ids.py': '832b20b5f97f8328150a6efd57847d9cc56378b703117da1bde65759f29c37ab', 'experiment.py': '518c606af5b1535d928e3bb3a7425a15db77525ea4bc944f2af74ffa0248ddb0', 'training_data.py': '6b16cc0d66276f782c9e839ec37fe6dbce703723f079ef706ef1be94a74cb684', 'reproducibility.py': 'ef59e1eb6453cfb149d11b6a4f2c7227d1aa8a8b7f354fb1a9a9b6e1590296aa', 'model.py': '34932db3ea2ac8b2bf3e0d2631e563dcbe22d686c3408a0d113cd499ff3630a1', 'data.py': '2be23fc5b5a5916c66074dc447143008719ee3c4b1eb2192b73d5f8b30be5989', 'metrics.py': '10d41722970c5472cfe9c3f09346ae977d95dab520fc8d72a4f26d3e7f935fbc', 'training.py': 'e567ed0eef9646479ce3cf50089ea024b1d12946d18e24444cde5dee00e1108d', 'evaluation/dimsim.py': '38712d92358e8378539935fd669f5e9a375193010007a3798faaea02c88e0195', 'evaluation/sampling.py': '39c473e643f1f30c62951d28d4e5b6004ed819256676ff351355d098ebeab0c8', 'evaluation/bundle.py': 'b0495dcba8fae1f13d79200f8e56faa4607fb84758cf899af1bed3c290bf876e', 'evaluation/representations.py': '8d41e607938a592559f77a4e3900ddda3c15975017d0589fbad59a634c43ddbb', 'evaluation/encoders.py': '85e1342b5a4028f850df26dd48683fb41ba569c38b13a786c5e74fa467f6e0de', 'evaluation/analysis.py': 'c06233981812e189e8b696c47fbe5d53d665a3d20bb611f0db15dffc12345577', 'evaluation/runner.py': 'a48b4e0478cf021a89f460641e94c559ca5f50f223e3dc7596a3a47e53d81e35'}, 'embedded_sha256': '58fbe8d0e00458bd36011e9dfc4d3e84abcbb531adc247320315fa3881a7b48c', 'generator_sha256': 'e62a8ed40d30b04792f6da7e9cdfa7e1d047300dbd4bb8e9c1cb559194f7988e'}


In [ ]:
import importlib.metadata
import subprocess
import sys

INSTALL_DEPENDENCIES = False
DEPENDENCIES = ["dimsim==0.2.2", "pypinyin==0.55.0", "numpy", "scipy", "pandas", "matplotlib"]
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *DEPENDENCIES])

for package in ("dimsim", "pypinyin", "numpy", "scipy", "pandas", "matplotlib"):
    print(package, importlib.metadata.version(package))
if importlib.metadata.version("dimsim") != "0.2.2" or importlib.metadata.version("pypinyin") != "0.55.0":
    raise RuntimeError("Pinned DIMSIM preparation dependencies do not match")


## Embedded `pinyin.py`


In [ ]:
"""Project TONE3 input contract backed by pinned pypinyin inventories."""
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import version
import re
from pypinyin.constants import PINYIN_DICT
from pypinyin.contrib.tone_convert import to_finals, to_initials, to_normal
PINYIN_CONTRACT_VERSION = 'pypinyin-tone3-1'

def _require_pypinyin_version():
    installed = version('pypinyin')
    if installed != '0.55.0':
        raise RuntimeError(f'pypinyin 0.55.0 required, found {installed}')
    return installed
_PYPINYIN_VERSION = _require_pypinyin_version()
VALID_SYLLABLES = frozenset((to_normal(reading, v_to_u=False).lower() for readings in PINYIN_DICT.values() for reading in readings.split(',')))
_INPUT_SYLLABLES = tuple(sorted((base for base in VALID_SYLLABLES if re.fullmatch('[a-z]+', base))))
_STRICT_COMPONENTS = tuple(((to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False)) for base in _INPUT_SYLLABLES))
INITIALS = sorted({initial for initial, final in _STRICT_COMPONENTS if final})
FINALS = sorted({final for _, final in _STRICT_COMPONENTS if final})
TONES = [str(tone) for tone in range(1, 6)]
PARSER_PROVENANCE = {'library': 'pypinyin', 'version': _PYPINYIN_VERSION, 'api': 'pypinyin.contrib.tone_convert', 'strict': True, 'input_format': 'lowercase-ascii-tone3', 'umlaut_storage': 'v', 'input_alias': 'u:', 'missing_tone': '5', 'empty_final': 'reject', 'inventory_source': 'pypinyin.constants.PINYIN_DICT', 'inventory_sha256': sha256('\n'.join(sorted(VALID_SYLLABLES)).encode('utf-8')).hexdigest(), 'vocabulary_sha256': sha256(repr((INITIALS, FINALS, TONES)).encode('utf-8')).hexdigest()}

@dataclass(frozen=True)
class PinyinComponents:
    initial: str
    final: str
    tone: str

def canonical_label(token: str, *, tone_convention='neutral_unmarked') -> str:
    """Accept one lowercase TONE3 syllable, allowing u: and unmarked neutral."""
    if tone_convention not in {'neutral_unmarked', 'explicit'}:
        raise ValueError(f'unsupported tone convention: {tone_convention!r}')
    if not isinstance(token, str) or not token:
        raise ValueError(f'Pinyin must be a non-empty string: {token!r}')
    if re.fullmatch('(?:[a-z]|u:)+[1-5]?', token) is None:
        raise ValueError(f'invalid Pinyin format (lowercase ASCII TONE3 required): {token!r}')
    canonical = token.replace('u:', 'v')
    if canonical[-1] not in TONES:
        if tone_convention == 'explicit':
            raise ValueError(f'explicit tone required: {token!r}')
        canonical += '5'
    if re.fullmatch('[a-z]+[1-5]', canonical) is None:
        raise ValueError(f'invalid canonical Pinyin format: {token!r}')
    base = canonical[:-1]
    if base not in VALID_SYLLABLES:
        raise ValueError(f'syllable absent from pypinyin dictionary: {token!r}')
    initial = to_initials(base, strict=True)
    final = to_finals(base, strict=True, v_to_u=False)
    if not final or initial not in INITIALS or final not in FINALS:
        raise ValueError(f'unsupported strict Pinyin components (non-empty final required): {token!r}')
    return canonical

def validate_canonical_label(token: str) -> str:
    """Reject noncanonical stored labels instead of silently repairing them."""
    if canonical_label(token, tone_convention='explicit') != token:
        raise ValueError(f'stored Pinyin label is not canonical: {token!r}')
    return token

def parse_components(token: str, *, tone_convention='neutral_unmarked') -> PinyinComponents:
    canonical = canonical_label(token, tone_convention=tone_convention)
    base = canonical[:-1]
    return PinyinComponents(to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False), canonical[-1])

def encode_components(components: PinyinComponents) -> tuple[int, int, int]:
    try:
        return (INITIALS.index(components.initial), FINALS.index(components.final), TONES.index(components.tone))
    except ValueError as exc:
        raise ValueError(f'components outside Pinyin vocabulary: {components!r}') from exc

def canonical_labels(tokens, *, tone_convention='neutral_unmarked'):
    return [canonical_label(token, tone_convention=tone_convention) for token in tokens]

def decompose(token: str, *, tone_convention='neutral_unmarked') -> tuple[int, int, int]:
    return encode_components(parse_components(token, tone_convention=tone_convention))


## Embedded `objectives.py`


In [ ]:
"""Objective identifiers and arbitrary finite acoustic loss weights."""
import math
import re
from numbers import Real
BASE_MODEL_IDS = ('syllable', 'components', 'acoustic')
COMBINED_PREFIX = 'components-acoustic-lambda-'

def combined_model_id(weight: Real) -> str:
    """Encode a finite coefficient without truncating its float precision."""
    if isinstance(weight, bool) or not isinstance(weight, Real):
        raise ValueError('lambda must be a finite real number')
    try:
        value = float(weight)
    except (OverflowError, ValueError) as exc:
        raise ValueError('lambda must be a finite real number') from exc
    if not math.isfinite(value):
        raise ValueError('lambda must be a finite real number')
    token = str(int(value)) if value.is_integer() and abs(value) < 1e+16 else repr(value).replace('e+', 'e')
    return COMBINED_PREFIX + token

def parse_model_id(model_id: str) -> tuple[str, float | None]:
    """Return the objective family and the combined acoustic coefficient."""
    if isinstance(model_id, str) and model_id in BASE_MODEL_IDS:
        return (model_id, None)
    if not isinstance(model_id, str) or not model_id.startswith(COMBINED_PREFIX):
        raise ValueError('unknown objective model_id')
    token = model_id[len(COMBINED_PREFIX):]
    if not re.fullmatch('[+-]?(?:\\d+(?:\\.\\d*)?|\\.\\d+)(?:[eE][+-]?\\d+)?', token):
        raise ValueError('combined model_id must contain a finite numeric lambda')
    weight = float(token)
    if not math.isfinite(weight):
        raise ValueError('lambda must be a finite real number')
    return ('components-acoustic', weight)

def canonical_model_id(model_id: str) -> str:
    family, weight = parse_model_id(model_id)
    return combined_model_id(weight) if family == 'components-acoustic' else family


## Embedded `artifacts.py`


In [ ]:
"""Small, deterministic artifact helpers for prepared datasets."""
import hashlib
import json
from pathlib import Path

def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def write_json(path: str | Path, value) -> None:
    Path(path).write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')

def write_jsonl(path: str | Path, rows) -> None:
    Path(path).write_text(''.join((json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in rows)), encoding='utf-8')

def read_jsonl(path: str | Path) -> list[dict]:
    result = []
    for number, line in enumerate(Path(path).read_text(encoding='utf-8').splitlines(), 1):
        try:
            value = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f'invalid JSONL at line {number}') from exc
        if not isinstance(value, dict):
            raise ValueError(f'JSONL row {number} is not an object')
        result.append(value)
    return result

def safe_artifact(root: str | Path, relative: str) -> Path:
    root = Path(root).resolve()
    rel = Path(relative)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError(f'unsafe artifact path: {relative}')
    path = (root / rel).resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'unsafe artifact path: {relative}')
    cursor = root
    for part in rel.parts:
        cursor = cursor / part
        if cursor.exists() and cursor.is_symlink():
            raise ValueError(f'symlink artifact path: {relative}')
    return path


## Embedded `data_contract.py`


In [ ]:
"""Shared prepared-data eligibility rules, independent of experiment splitting."""
import hashlib
import json
DATA_CONTRACT_VERSION = 'prepared-1'
FILTERS = ('nonempty', 'characters', 'alignment', 'pinyin', 'ids', 'acoustic')

def sample_id(text, labels):
    return hashlib.sha256(json.dumps([text, labels], ensure_ascii=False, separators=(',', ':')).encode('utf-8')).hexdigest()

def is_han(character):
    code = ord(character)
    return 13312 <= code <= 19903 or 19968 <= code <= 40959 or 63744 <= code <= 64255 or (131072 <= code <= 205743)

def pair_reason(text, labels, ids_mapping, acoustic, *, stage=None):
    """Return the first failing stage; a supplied stage checks only that rule."""
    stages = FILTERS if stage is None else (stage,)
    for current in stages:
        if current == 'nonempty':
            if not isinstance(text, str) or not text or (not labels):
                return current
        elif current == 'characters':
            if any((not is_han(ch) for ch in text)):
                return current
        elif current == 'alignment':
            if len(text) != len(labels):
                return current
        elif current == 'pinyin':
            try:
                for label in labels:
                    decompose(label)
            except (ValueError, TypeError, IndexError):
                return current
        elif current == 'ids':
            if any((ch not in ids_mapping for ch in text)):
                return current
        elif current == 'acoustic':
            if any((label not in acoustic for label in labels)):
                return current
        else:
            raise ValueError(f'unknown filter stage: {current}')
    return None

def validate_wubi(mapping, *, alphabet='abcdefghijklmnopqrstuvwxyz0123456789'):
    """Validate the separately maintained historical inference mapping."""
    if not isinstance(mapping, dict) or not mapping:
        raise ValueError('mapping must be a non-empty dict')
    for key, value in mapping.items():
        if not isinstance(key, str) or len(key) != 1 or (not isinstance(value, str)) or (not value) or any((symbol not in alphabet for symbol in value)):
            raise ValueError(f'invalid mapping entry: {key!r}')
    return dict(sorted(mapping.items()))

def validate_acoustic(mapping, *, expected_keys=None):
    import numpy as np
    if not isinstance(mapping, dict) or not mapping:
        raise ValueError('acoustic mapping must be a non-empty dict')
    if any((not isinstance(key, str) or not key for key in mapping)):
        raise ValueError('invalid acoustic key')
    if expected_keys is not None and set(mapping) != set(expected_keys):
        raise ValueError('acoustic keys disagree with inventory')
    dimension = None
    for key, value in sorted(mapping.items()):
        vector = np.asarray(value, dtype=np.float32)
        if vector.ndim != 1 or not vector.size:
            raise ValueError(f'invalid acoustic vector: {key}')
        if dimension is None:
            dimension = vector.size
        if vector.size != dimension or not np.isfinite(vector).all() or np.linalg.norm(vector.astype(np.float64)) <= 1e-12:
            raise ValueError(f'invalid acoustic vector: {key}')
    return dimension


## Embedded `ids.py`


In [ ]:
"""Pinned, deterministic IDS parsing and token mapping."""
import hashlib
import json
from pathlib import Path
IDS_SCHEMA_VERSION = 1
IDS_RULES_VERSION = 'ids-prefix-1'
PAD = '[PAD]'
UNKNOWN = '？'
OPERATORS = {chr(code) for code in range(12272, 12284)}
TERNARY_OPERATORS = {'⿲', '⿳'}
UNSUPPORTED_OPERATORS = {chr(code) for code in range(12284, 12288)} | {'\u31ef'}
EXPECTED_SNAPSHOT = '618dbab8a8ddefb958763c8b4afbaa741a4460de'
EXPECTED_LICENSE = 'LGPL-3.0-or-later'
EXPECTED_ATTRIBUTION = 'Make Me a Hanzi; dictionary.txt derived from Unihan and CJKlib'

def parse_expression(expression: str) -> tuple[str, ...]:
    if not isinstance(expression, str) or not expression:
        raise ValueError('IDS decomposition must be a non-empty string')
    if any((char in UNSUPPORTED_OPERATORS for char in expression)):
        raise ValueError('unsupported IDS operator')
    cursor = 0

    def parse_node() -> None:
        nonlocal cursor
        if cursor >= len(expression):
            raise ValueError('IDS expression has too few operands')
        symbol = expression[cursor]
        cursor += 1
        if symbol in OPERATORS:
            arity = 3 if symbol in TERNARY_OPERATORS else 2
            for _ in range(arity):
                parse_node()
    parse_node()
    if cursor != len(expression):
        raise ValueError('IDS expression has trailing operands')
    return tuple(expression)

def read_dictionary(path: str | Path) -> dict[str, dict]:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    mapping: dict[str, dict] = {}
    with path.open('r', encoding='utf-8', newline='') as handle:
        for line_number, line in enumerate(handle, 1):
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'invalid IDS JSONL at line {line_number}') from exc
            if not isinstance(row, dict):
                raise ValueError(f'IDS line {line_number} must be an object')
            character, decomposition = (row.get('character'), row.get('decomposition'))
            if not isinstance(character, str) or len(character) != 1:
                raise ValueError(f'IDS line {line_number} character must be one code point')
            if character in mapping:
                raise ValueError(f'duplicate IDS character: {character}')
            if not isinstance(decomposition, str) or not decomposition:
                raise ValueError(f'empty or invalid IDS decomposition for {character}')
            parsed = parse_expression(decomposition)
            if UNKNOWN in decomposition:
                tokens, mode = ((character,), 'character_fallback')
            else:
                tokens, mode = (parsed, 'decomposition')
            mapping[character] = {'decomposition': decomposition, 'tokens': list(tokens), 'mode': mode}
    if not mapping:
        raise ValueError('IDS dictionary is empty')
    return dict(sorted(mapping.items(), key=lambda item: tuple(map(ord, item[0]))))

def validate_pin(spec: dict) -> tuple[Path, str]:
    expected = {'path', 'sha256', 'snapshot_commit', 'license', 'attribution'}
    if not isinstance(spec, dict) or set(spec) != expected:
        raise ValueError('IDS pin must contain path, sha256, snapshot_commit, license, attribution')
    path = Path(spec['path']).resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    digest = sha256_file(path)
    if digest != spec['sha256']:
        raise ValueError('IDS dictionary checksum mismatch')
    if spec['snapshot_commit'] != EXPECTED_SNAPSHOT:
        raise ValueError('IDS snapshot commit mismatch')
    if spec['license'] != EXPECTED_LICENSE or not isinstance(spec['attribution'], str) or (not spec['attribution'].strip()):
        raise ValueError('IDS license or attribution mismatch')
    return (path, digest)

def build_mapping(spec: dict, *, dictionary_path: str | Path | None=None) -> tuple[dict, dict, dict]:
    if dictionary_path is None:
        path, digest = validate_pin(spec)
        records_path = path
    else:
        expected = {'path', 'sha256', 'snapshot_commit', 'license', 'attribution'}
        if not isinstance(spec, dict) or set(spec) != expected:
            raise ValueError('IDS pin must contain path, sha256, snapshot_commit, license, attribution')
        records_path = Path(dictionary_path).resolve()
        if not records_path.is_file() or sha256_file(records_path) != spec['sha256']:
            raise ValueError('bundled IDS dictionary checksum mismatch')
        if spec['snapshot_commit'] != EXPECTED_SNAPSHOT or spec['license'] != EXPECTED_LICENSE or (not str(spec['attribution']).strip()):
            raise ValueError('IDS source provenance mismatch')
        path, digest = (spec['path'], spec['sha256'])
    records = read_dictionary(records_path)
    tokens = sorted({token for record in records.values() for token in record['tokens']}, key=ord)
    vocabulary = [PAD, *tokens]
    stoi = {token: index for index, token in enumerate(vocabulary)}
    encoded = {character: {**record, 'ids': [stoi[token] for token in record['tokens']]} for character, record in records.items()}
    provenance = {'path': str(path), 'sha256': digest, 'snapshot_commit': spec['snapshot_commit'], 'license': spec['license'], 'attribution': spec['attribution']}
    mapping_artifact = {'schema_version': IDS_SCHEMA_VERSION, 'rules_version': IDS_RULES_VERSION, 'provenance': provenance, 'characters': encoded}
    vocabulary_artifact = {'schema_version': IDS_SCHEMA_VERSION, 'rules_version': IDS_RULES_VERSION, 'tokens': vocabulary, 'stoi': stoi}
    return (mapping_artifact, vocabulary_artifact, provenance)

def encode_text(text: str, mapping: dict, vocabulary: dict) -> tuple[list[list[int]], list[int], str | None]:
    sequences, lengths = ([], [])
    for character in text:
        if character not in mapping['characters']:
            raise ValueError(f'character is absent from IDS mapping: {character}')
        record = mapping['characters'][character]
        if len(record['ids']) != len(record['tokens']) or any((type(i) is not int or i <= 0 or i >= len(vocabulary['tokens']) or (vocabulary['tokens'][i] != token) for token, i in zip(record['tokens'], record['ids']))):
            raise ValueError(f'invalid IDS token ID in mapping: {character}')
        sequences.append(list(record['ids']))
        lengths.append(len(record['ids']))
    return (sequences, lengths, None)

def content_hash(value: dict) -> str:
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()


## Embedded `experiment.py`


In [ ]:
"""Portable dataset-only experiment bundles with no ML runtime dependency."""
import copy
import json
import math
import os
import pickle
import random
import shutil
import tempfile
from pathlib import Path, PureWindowsPath
import numpy as np
PREPARED_SHA256 = 'c562722d2a866f4f1904c746c86c5863a310a66ee3875d86fc36f254fd2d7c10'
SCHEMA_VERSION = 3
DEFAULT_MODEL_IDS = ['syllable', 'components', 'acoustic', 'components-acoustic-lambda-1', 'components-acoustic-lambda-10']
REQUIRED_FILES = {'samples.jsonl', 'ids/mapping.json', 'ids/vocabulary.json', 'ids/dictionary.jsonl', 'ids/COPYING', 'ids/LGPL', 'acoustic_targets.npy', 'vocab.json', 'splits.json', 'config.json', 'provenance/prepared_manifest.json', 'provenance/processed_manifest.json', 'provenance/processed_config.json', 'provenance/acoustic_manifest.json', 'provenance/sound_inventory.jsonl', 'provenance/prepared_corpus.jsonl', 'provenance/prepared_acoustic_targets.pkl'}
_ARCH = {'embedding_dim': 64, 'ids_bigru_hidden_per_direction': 32, 'ids_bigru_layers': 1, 'sentence_bilstm_hidden_per_direction': 64, 'sentence_bilstm_layers': 1, 'context_dim': 128, 'classification_hidden_dim': 64, 'classification_dropout': 0.5}
_TRAIN = {'optimizer': 'Adam', 'learning_rate': 0.001, 'weight_decay': 0.0, 'batch_size': 32, 'epochs': 20, 'scheduler': None, 'early_stopping': False, 'gradient_clipping': None, 'cosine_epsilon': 1e-08}

def default_experiment_config(acoustic_dim=1024):
    """Return a fresh config with the agreed protocol and current input pin."""
    return {'schema_version': SCHEMA_VERSION, 'prepared_manifest_sha256': PREPARED_SHA256, 'split_ratios': [0.8, 0.1, 0.1], 'split_seed': 0, 'training_seeds': [0, 1, 2, 3, 4], 'architecture': {**_ARCH, 'acoustic_dim': acoustic_dim}, 'training': copy.deepcopy(_TRAIN), 'model_ids': list(DEFAULT_MODEL_IDS)}

def _validate_config(value, acoustic_dim):
    expected = default_experiment_config(acoustic_dim)
    if not isinstance(value, dict) or set(value) != set(expected):
        raise ValueError('invalid experiment config keys')
    cfg = copy.deepcopy(value)

    def exact_type_value(actual, wanted):
        if type(actual) is not type(wanted):
            return False
        if isinstance(wanted, dict):
            return set(actual) == set(wanted) and all((exact_type_value(actual[k], wanted[k]) for k in wanted))
        if isinstance(wanted, list):
            return len(actual) == len(wanted) and all((exact_type_value(a, b) for a, b in zip(actual, wanted)))
        return actual == wanted
    if type(cfg['schema_version']) is not int or cfg['schema_version'] != SCHEMA_VERSION or (not exact_type_value(cfg['architecture'], expected['architecture'])) or (not exact_type_value(cfg['training'], _TRAIN)):
        raise ValueError('experiment config conflicts with the training protocol')
    model_ids = cfg['model_ids']
    if not isinstance(model_ids, list) or not model_ids:
        raise ValueError('model_ids must be a nonempty list')
    canonical_ids = [canonical_model_id(model_id) for model_id in model_ids]
    if len(set(canonical_ids)) != len(canonical_ids):
        raise ValueError('model_ids must identify distinct objectives')
    pin = cfg['prepared_manifest_sha256']
    if not isinstance(pin, str) or len(pin) != 64 or any((c not in '0123456789abcdef' for c in pin)):
        raise ValueError('prepared_manifest_sha256 must be a lowercase SHA-256')
    ratios = cfg['split_ratios']
    if not isinstance(ratios, list) or len(ratios) != 3 or any((type(x) not in (int, float) or not math.isfinite(x) or x <= 0 for x in ratios)) or (not math.isclose(sum(ratios), 1.0, rel_tol=0.0, abs_tol=1e-09)):
        raise ValueError('split_ratios must contain three finite positive values summing to one')
    if type(cfg['split_seed']) is not int or cfg['split_seed'] != 0:
        raise ValueError('split_seed must be the fixed protocol value 0')
    seeds = cfg['training_seeds']
    if not isinstance(seeds, list) or len(seeds) != 5 or any((type(s) is not int for s in seeds)) or (seeds != [0, 1, 2, 3, 4]):
        raise ValueError('training_seeds must be the fixed protocol values [0, 1, 2, 3, 4]')
    return cfg

def _split_ids(ids, seed, ratios):
    values = sorted(ids)
    random.Random(seed).shuffle(values)
    n = len(values)
    train_end = int(n * ratios[0])
    validation_end = train_end + int(n * ratios[1])
    result = {'train': values[:train_end], 'validation': values[train_end:validation_end], 'test': values[validation_end:]}
    if any((not group for group in result.values())):
        raise ValueError('train, validation, and test splits must all be nonempty')
    return result

def _safe_id(value):
    return isinstance(value, str) and bool(value) and (value not in {'.', '..'}) and ('/' not in value) and ('\\' not in value) and (Path(value).name == value) and (PureWindowsPath(value).name == value)

def prepare_experiment(experiment_id, *, prepared_root, output_root=None, config=None):
    """Create, validate and atomically publish a portable dataset bundle."""
    if not _safe_id(experiment_id):
        raise ValueError('experiment_id must be one safe path component')
    src = Path(prepared_root).resolve()
    output = Path(output_root).resolve() if output_root else src.parent.parent / 'experiments'
    target = (output / experiment_id).resolve()
    if target.parent != output:
        raise ValueError('experiment path escapes output_root')
    if target.exists():
        raise FileExistsError(f'experiment already exists: {target}')
    prepared_manifest = validate_prepared(src)
    pin = sha256_file(src / 'manifest.json')
    acoustic_bytes = (src / 'acoustic/targets.pkl').read_bytes()
    if sha256_file(src / 'acoustic/targets.pkl') != prepared_manifest['artifacts']['acoustic/targets.pkl']:
        raise ValueError('prepared acoustic target hash mismatch')
    acoustic = pickle.loads(acoustic_bytes)
    keys = sorted(acoustic)
    if not keys or len(keys) != len(set(keys)):
        raise ValueError('acoustic target inventory is empty or duplicated')
    matrix = np.stack([np.asarray(acoustic[k], dtype=np.float32) for k in keys])
    if matrix.ndim != 2 or not np.isfinite(matrix).all() or np.any(np.linalg.norm(matrix.astype(np.float64), axis=1) <= 1e-12):
        raise ValueError('acoustic targets must be finite nonzero vectors')
    cfg = _validate_config(config if config is not None else default_experiment_config(matrix.shape[1]), matrix.shape[1])
    if pin != cfg['prepared_manifest_sha256']:
        raise ValueError('prepared manifest checksum does not match config pin')
    rows = read_jsonl(src / 'corpus.jsonl')
    key_ids = {label: i for i, label in enumerate(keys)}
    for row in rows:
        labels = row['labels']
        component_ids = [list(decompose(validate_canonical_label(label))) for label in labels]
        row['component_labels'] = component_ids
        row['syllable_ids'] = [key_ids[label] for label in labels]
        row['acoustic_ids'] = list(row['syllable_ids'])
    output.mkdir(parents=True, exist_ok=True)
    if target.exists():
        raise FileExistsError(f'experiment already exists: {target}')
    temp = Path(tempfile.mkdtemp(prefix=f'.{experiment_id}.', dir=output))
    try:
        (temp / 'ids').mkdir()
        (temp / 'provenance').mkdir()
        with (temp / 'samples.jsonl').open('w', encoding='utf-8', newline='\n') as handle:
            for row in rows:
                handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')
        for name in ('mapping.json', 'vocabulary.json', 'dictionary.jsonl', 'COPYING', 'LGPL'):
            shutil.copyfile(src / 'ids' / name, temp / 'ids' / name)
        for source, relative in ((src / 'manifest.json', 'prepared_manifest.json'), (src / 'reports/processed_corpus_manifest.json', 'processed_manifest.json'), (src / 'reports/processed_corpus_config.json', 'processed_config.json'), (src / 'reports/acoustic_bundle_manifest.json', 'acoustic_manifest.json'), (src / 'reports/sound_inventory.jsonl', 'sound_inventory.jsonl'), (src / 'corpus.jsonl', 'prepared_corpus.jsonl')):
            shutil.copyfile(source, temp / 'provenance' / relative)
        (temp / 'provenance/prepared_acoustic_targets.pkl').write_bytes(acoustic_bytes)
        np.save(temp / 'acoustic_targets.npy', matrix, allow_pickle=False)
        vocabulary = json.loads((src / 'ids/vocabulary.json').read_text(encoding='utf-8'))
        mapping = json.loads((src / 'ids/mapping.json').read_text(encoding='utf-8'))
        write_json(temp / 'vocab.json', {'ids': vocabulary, 'ids_mapping': mapping, 'acoustic_tokens': keys, 'syllables': keys, 'labels': {'initials': INITIALS, 'finals': FINALS, 'tones': TONES}})
        write_json(temp / 'splits.json', _split_ids([row['sample_id'] for row in rows], cfg['split_seed'], cfg['split_ratios']))
        write_json(temp / 'config.json', cfg)
        artifacts = {str(p.relative_to(temp)).replace('\\', '/'): sha256_file(p) for p in temp.rglob('*') if p.is_file()}
        provenance = {name: sha256_file(temp / name) for name in sorted(artifacts) if name.startswith('provenance/')}
        write_json(temp / 'manifest.json', {'bundle_type': 'experiment_dataset', 'schema_version': SCHEMA_VERSION, 'experiment_id': experiment_id, 'pinyin_contract': PINYIN_CONTRACT_VERSION, 'parser_provenance': PARSER_PROVENANCE, 'ids_rules': IDS_RULES_VERSION, 'prepared_manifest_sha256': pin, 'prepared_dataset_id': prepared_manifest['dataset_id'], 'provenance': provenance, 'counts': {'samples': len(rows), 'positions': sum((row['sentence_length'] for row in rows)), 'acoustic_targets': len(keys), 'acoustic_dim': matrix.shape[1]}, 'artifacts': artifacts})
        validate_experiment(temp)
        os.rename(temp, target)
        return target
    except Exception:
        shutil.rmtree(temp, ignore_errors=True)
        raise

def _validate_semantics(root, manifest, config, rows, splits, vocab, matrix):
    if manifest.get('pinyin_contract') != PINYIN_CONTRACT_VERSION or manifest.get('parser_provenance') != PARSER_PROVENANCE or manifest.get('ids_rules') != IDS_RULES_VERSION:
        raise ValueError('unsupported experiment provenance contract')
    pin = manifest.get('prepared_manifest_sha256')
    if sha256_file(root / 'provenance/prepared_manifest.json') != pin or pin != config['prepared_manifest_sha256']:
        raise ValueError('prepared provenance pin mismatch')
    prepared = json.loads((root / 'provenance/prepared_manifest.json').read_text(encoding='utf-8'))
    if prepared.get('dataset_id') != manifest.get('prepared_dataset_id'):
        raise ValueError('prepared dataset identity mismatch')
    processed_path = root / 'provenance/processed_manifest.json'
    processed = json.loads(processed_path.read_text(encoding='utf-8'))
    acoustic_path = root / 'provenance/acoustic_manifest.json'
    acoustic_manifest = json.loads(acoustic_path.read_text(encoding='utf-8'))
    inventory_path = root / 'provenance/sound_inventory.jsonl'
    inventory = read_jsonl(inventory_path)
    inventory_hash = sha256_file(inventory_path)
    corpus_path = root / 'provenance/prepared_corpus.jsonl'
    acoustic_targets_path = root / 'provenance/prepared_acoustic_targets.pkl'
    if prepared.get('artifacts', {}).get('corpus.jsonl') != sha256_file(corpus_path) or prepared.get('artifacts', {}).get('acoustic/targets.pkl') != sha256_file(acoustic_targets_path) or prepared.get('processed_corpus_manifest_sha256') != sha256_file(processed_path) or (prepared.get('acoustic_bundle_manifest_sha256') != sha256_file(acoustic_path)) or (processed.get('sources', {}).get('sound_inventory_sha256') != inventory_hash) or (acoustic_manifest.get('inventory_sha256') != inventory_hash) or (processed.get('artifacts', {}).get('config.json') != sha256_file(root / 'provenance/processed_config.json')):
        raise ValueError('copied upstream provenance does not agree')
    if prepared.get('artifacts', {}).get('ids/mapping.json') != sha256_file(root / 'ids/mapping.json'):
        raise ValueError('prepared IDS mapping provenance mismatch')
    if prepared.get('artifacts', {}).get('ids/vocabulary.json') != sha256_file(root / 'ids/vocabulary.json'):
        raise ValueError('prepared IDS vocabulary provenance mismatch')
    if prepared.get('artifacts', {}).get('ids/dictionary.jsonl') != sha256_file(root / 'ids/dictionary.jsonl'):
        raise ValueError('prepared IDS dictionary provenance mismatch')
    for name, digest in manifest['provenance'].items():
        if sha256_file(safe_artifact(root, name)) != digest:
            raise ValueError(f'provenance checksum mismatch: {name}')
    if set(manifest['provenance']) != {n for n in manifest['artifacts'] if n.startswith('provenance/')}:
        raise ValueError('provenance file set mismatch')
    original_vocab = json.loads((root / 'ids/vocabulary.json').read_text(encoding='utf-8'))
    original_mapping = json.loads((root / 'ids/mapping.json').read_text(encoding='utf-8'))
    bundled_vocab = vocab['ids']
    if bundled_vocab != original_vocab or vocab['ids_mapping'] != original_mapping:
        raise ValueError('duplicated IDS mapping or vocabulary mismatch')
    if original_vocab.get('tokens', [None])[0] != '[PAD]' or original_vocab.get('stoi', {}).get('[PAD]') != 0:
        raise ValueError('IDS PAD token must have ID zero')
    ids_cfg = prepared.get('ids_provenance')
    rebuilt_mapping, rebuilt_vocab, _ = build_mapping(ids_cfg, dictionary_path=root / 'ids/dictionary.jsonl')
    if rebuilt_mapping != original_mapping or rebuilt_vocab != original_vocab:
        raise ValueError('bundled IDS mapping or vocabulary fails reconstruction')
    tokens = vocab.get('acoustic_tokens')
    if not isinstance(tokens, list) or not tokens or tokens != sorted(tokens) or (len(tokens) != len(set(tokens))) or (vocab.get('syllables') != tokens) or (vocab.get('labels') != {'initials': INITIALS, 'finals': FINALS, 'tones': TONES}):
        raise ValueError('invalid acoustic or syllable vocabulary')
    inventory_labels = [row.get('label') for row in inventory]
    if sorted(inventory_labels) != tokens or len(inventory_labels) != len(set(inventory_labels)):
        raise ValueError('acoustic inventory does not match full syllable vocabulary')
    key_ids = {label: i for i, label in enumerate(tokens)}
    for label in tokens:
        validate_canonical_label(label)
    if matrix.shape != (len(tokens), config['architecture']['acoustic_dim']) or not np.isfinite(matrix).all() or np.any(np.linalg.norm(matrix.astype(np.float64), axis=1) <= 1e-12):
        raise ValueError('invalid acoustic target matrix')
    with acoustic_targets_path.open('rb') as handle:
        pinned_acoustic = pickle.load(handle)
    pinned_keys = sorted(pinned_acoustic)
    pinned_matrix = np.stack([np.asarray(pinned_acoustic[k], dtype=np.float32) for k in pinned_keys])
    if pinned_keys != tokens or pinned_matrix.shape != matrix.shape or (not np.array_equal(pinned_matrix, matrix)):
        raise ValueError('acoustic matrix does not reproduce pinned prepared targets')
    mapping_chars = original_mapping.get('characters', {})
    id_count = len(original_vocab['tokens'])
    seen_ids = []
    seen_sources = set()
    for row in rows:
        required = {'component_lengths', 'ids_sequences', 'labels', 'positions', 'sample_id', 'sentence_length', 'source_ids', 'text', 'component_labels', 'syllable_ids', 'acoustic_ids'}
        if set(row) != required:
            raise ValueError('sample keys do not match experiment schema')
        text, labels = (row['text'], row['labels'])
        n = len(text) if isinstance(text, str) else -1
        if n <= 0 or type(row['sentence_length']) is not int or row['sentence_length'] != n:
            raise ValueError('invalid sample text or sentence length')
        if row['sample_id'] != sample_id(text, labels):
            raise ValueError('sample ID does not match text and labels')
        if not isinstance(row['positions'], list) or any((type(pos) is not int for pos in row['positions'])) or row['positions'] != list(range(n)) or (not isinstance(labels, list)) or (len(labels) != n) or (not isinstance(row['ids_sequences'], list)) or (len(row['ids_sequences']) != n) or (not isinstance(row['component_lengths'], list)) or (len(row['component_lengths']) != n):
            raise ValueError('sample alignment mismatch')
        if not isinstance(row['source_ids'], list) or not row['source_ids'] or any((not isinstance(s, str) or not s for s in row['source_ids'])) or (len(set(row['source_ids'])) != len(row['source_ids'])):
            raise ValueError('invalid sample source IDs')
        if seen_sources.intersection(row['source_ids']):
            raise ValueError('source IDs occur in more than one sample')
        seen_sources.update(row['source_ids'])
        expected_components, expected_syllables, expected_sequences = ([], [], [])
        for char, label, sequence, length in zip(text, labels, row['ids_sequences'], row['component_lengths']):
            canonical = validate_canonical_label(label)
            expected_components.append(list(decompose(canonical)))
            if canonical not in key_ids:
                raise ValueError(f'missing acoustic target for label: {canonical}')
            expected_syllables.append(key_ids[canonical])
            if not isinstance(sequence, list) or not sequence or type(length) is not int or (length != len(sequence)):
                raise ValueError('invalid IDS sequence or component length')
            if any((type(token) is not int or token <= 0 or token >= id_count for token in sequence)):
                raise ValueError('IDS token ID outside vocabulary or padding')
            char_entry = mapping_chars.get(char)
            if char_entry is None or sequence != char_entry.get('ids'):
                raise ValueError('IDS sequence does not match character mapping')
            expected_sequences.append(sequence)
        if not isinstance(row['component_labels'], list) or any((not isinstance(x, list) or any((type(i) is not int for i in x)) for x in row['component_labels'])) or row['component_labels'] != expected_components:
            raise ValueError('component targets mismatch canonical Pinyin')
        if any((type(i) is not int for i in row['syllable_ids'] + row['acoustic_ids'])):
            raise ValueError('syllable/acoustic IDs must be integers')
        if row['syllable_ids'] != expected_syllables or row['acoustic_ids'] != expected_syllables:
            raise ValueError('syllable/acoustic targets mismatch canonical Pinyin')
        seen_ids.append(row['sample_id'])
    if seen_ids != sorted(seen_ids) or len(seen_ids) != len(set(seen_ids)):
        raise ValueError('sample ID order or uniqueness invalid')
    source_rows = read_jsonl(corpus_path)
    base_fields = {'component_lengths', 'ids_sequences', 'labels', 'positions', 'sample_id', 'sentence_length', 'source_ids', 'text'}
    projected_rows = [{key: row[key] for key in base_fields} for row in rows]
    if projected_rows != source_rows:
        raise ValueError('experiment samples do not match pinned prepared corpus')
    if set(splits) != {'train', 'validation', 'test'}:
        raise ValueError('split names are invalid')
    split_ids = [sid for name in ('train', 'validation', 'test') for sid in splits[name]]
    if any((not splits[k] for k in splits)) or len(split_ids) != len(set(split_ids)) or set(split_ids) != set(seen_ids):
        raise ValueError('split membership is incomplete or duplicated')
    if splits != _split_ids(seen_ids, config['split_seed'], config['split_ratios']):
        raise ValueError('splits do not match deterministic config')
    counts = manifest.get('counts')
    expected_counts = {'samples': len(rows), 'positions': sum((row['sentence_length'] for row in rows)), 'acoustic_targets': len(tokens), 'acoustic_dim': matrix.shape[1]}
    if counts != expected_counts:
        raise ValueError('experiment counts mismatch')

def validate_experiment(path):
    root = Path(path).resolve()
    manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
    if set(manifest) != {'bundle_type', 'schema_version', 'experiment_id', 'pinyin_contract', 'parser_provenance', 'ids_rules', 'prepared_manifest_sha256', 'prepared_dataset_id', 'provenance', 'counts', 'artifacts'}:
        raise ValueError('invalid experiment manifest keys')
    if manifest.get('bundle_type') != 'experiment_dataset' or manifest.get('schema_version') != SCHEMA_VERSION or (not _safe_id(manifest.get('experiment_id'))):
        raise ValueError('unsupported experiment bundle')
    artifacts = manifest['artifacts']
    if set(artifacts) != REQUIRED_FILES:
        raise ValueError('experiment required artifact set mismatch')
    actual = {str(p.relative_to(root)).replace('\\', '/') for p in root.rglob('*') if p.is_file() and p.name != 'manifest.json'}
    if actual != set(artifacts):
        raise ValueError('experiment contains untracked or missing artifacts')
    for relative, digest in artifacts.items():
        p = safe_artifact(root, relative)
        if not p.is_file() or not isinstance(digest, str) or sha256_file(p) != digest:
            raise ValueError(f'artifact checksum mismatch: {relative}')
    config = json.loads((root / 'config.json').read_text(encoding='utf-8'))
    matrix = np.load(root / 'acoustic_targets.npy', allow_pickle=False)
    config = _validate_config(config, matrix.shape[1] if matrix.ndim == 2 else -1)
    if manifest.get('prepared_manifest_sha256') != config['prepared_manifest_sha256']:
        raise ValueError('manifest and config input pins disagree')
    vocabulary = json.loads((root / 'vocab.json').read_text(encoding='utf-8'))
    splits = json.loads((root / 'splits.json').read_text(encoding='utf-8'))
    rows = read_jsonl(root / 'samples.jsonl')
    _validate_semantics(root, manifest, config, rows, splits, vocabulary, matrix)
    return manifest

def load_experiment(path):
    """Load a portable bundle; samples are keyed by sample_id."""
    manifest = validate_experiment(path)
    root = Path(path).resolve()
    return {'root': root, 'manifest': manifest, 'config': json.loads((root / 'config.json').read_text(encoding='utf-8')), 'vocab': json.loads((root / 'vocab.json').read_text(encoding='utf-8')), 'splits': json.loads((root / 'splits.json').read_text(encoding='utf-8')), 'samples': {row['sample_id']: row for row in read_jsonl(root / 'samples.jsonl')}, 'acoustic_targets': np.load(root / 'acoustic_targets.npy', allow_pickle=False)}


## Embedded `training_data.py`


In [ ]:
"""Pure Pinyin-inventory helpers shared by training diagnostics."""
from functools import lru_cache

@lru_cache(maxsize=1)
def valid_initial_final_pairs() -> frozenset[tuple[int, int]]:
    """Return pairs represented by labels accepted by the strict Pinyin API."""
    pairs = set()
    for base in sorted(VALID_SYLLABLES):
        try:
            canonical = canonical_label(base + '1', tone_convention='explicit')
            initial_id, final_id, _ = decompose(canonical, tone_convention='explicit')
        except (ValueError, TypeError, IndexError):
            continue
        pairs.add((initial_id, final_id))
    if not pairs:
        raise RuntimeError('strict Pinyin inventory contains no valid initial/final pairs')
    return frozenset(pairs)


## Embedded `reproducibility.py`


In [ ]:
"""Deterministic training seeds and serializable RNG snapshots."""
import random
import numpy as np
import torch

def seed_everything(seed: int, deterministic: bool=True) -> None:
    if type(seed) is not int or seed < 0:
        raise ValueError('seed must be a nonnegative integer')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)
        if hasattr(torch.backends, 'cudnn'):
            torch.backends.cudnn.benchmark = False

def make_generator(seed: int, stream: int=0) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(int(seed) + 1000003 * int(stream))
    return generator

def rng_snapshot(loader_generator: torch.Generator | None=None) -> dict:
    numpy_state = np.random.get_state()
    state = {'python': list(random.getstate()), 'numpy': {'algorithm': numpy_state[0], 'keys': numpy_state[1].tolist(), 'position': int(numpy_state[2]), 'has_gauss': int(numpy_state[3]), 'cached_gaussian': float(numpy_state[4])}, 'torch_cpu': torch.get_rng_state(), 'torch_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}
    if loader_generator is not None:
        state['dataloader'] = loader_generator.get_state()
    return state


## Embedded `model.py`


In [ ]:
"""Shared IDS and sentence encoders with condition-specific readout heads."""
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class IDSCharacterEncoder(nn.Module):

    def __init__(self, vocab_size: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.bigru = nn.GRU(64, 32, num_layers=1, bidirectional=True, batch_first=True)

    def forward(self, token_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        valid = lengths.reshape(-1) > 0
        if not bool(valid.any()):
            return self.embedding.weight.new_zeros((lengths.numel(), 64))
        selected_ids = token_ids.reshape(-1, token_ids.shape[-1])[valid]
        selected_lengths = lengths.reshape(-1)[valid].to(device='cpu', dtype=torch.long)
        embedded = self.embedding(selected_ids)
        packed = pack_padded_sequence(embedded, selected_lengths, batch_first=True, enforce_sorted=False)
        _, hidden = self.bigru(packed)
        character_vectors = torch.cat((hidden[-2], hidden[-1]), dim=-1)
        result = self.embedding.weight.new_zeros((lengths.numel(), 64))
        return result.index_copy(0, valid.nonzero(as_tuple=False).flatten(), character_vectors).reshape(*lengths.shape, 64)

class SharedEncoder(nn.Module):
    """Encode variable-length IDS for each character, then full sentence context."""
    context_dim = 128

    def __init__(self, ids_vocab_size: int):
        super().__init__()
        self.ids = IDSCharacterEncoder(ids_vocab_size)
        self.sentence = nn.LSTM(64, 64, num_layers=1, bidirectional=True, batch_first=True)

    def encode(self, batch: dict) -> tuple[torch.Tensor, torch.Tensor]:
        token_ids = batch['ids_sequences']
        component_lengths = batch['component_lengths']
        sentence_lengths = batch['sentence_lengths'].to(dtype=torch.long)
        mask = batch['char_mask'].to(dtype=torch.bool)
        if token_ids.ndim != 3 or component_lengths.shape != token_ids.shape[:2] or mask.shape != token_ids.shape[:2]:
            raise ValueError('invalid padded IDS batch shapes')
        if not torch.equal(mask, component_lengths > 0):
            raise ValueError('character mask and IDS lengths disagree')
        if not bool((sentence_lengths > 0).all()) or not torch.equal(sentence_lengths.cpu(), mask.sum(1).cpu()):
            raise ValueError('sentence lengths disagree with character mask')
        char_vectors = self.ids(token_ids, component_lengths)
        packed = pack_padded_sequence(char_vectors, sentence_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.sentence(packed)
        contextual, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=token_ids.shape[1])
        return (contextual, mask)

    def forward(self, batch: dict) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encode(batch)

class ClassifierHead(nn.Module):

    def __init__(self, output_dim: int):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, output_dim))

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        return self.layers(hidden)

class TrainingModel(nn.Module):

    def __init__(self, encoder: SharedEncoder, model_id: str, vocab: dict, acoustic_dim: int):
        super().__init__()
        model_id = canonical_model_id(model_id)
        family, weight = parse_model_id(model_id)
        self.encoder = encoder
        self.model_id = model_id
        self.lambda_cosine = weight
        self.context_dim = 128
        labels = vocab['labels']
        self.component_heads = nn.ModuleDict()
        if family in {'components', 'components-acoustic'}:
            for name in ('initials', 'finals', 'tones'):
                self.component_heads[name] = ClassifierHead(len(labels[name]))
        self.syllable_head = ClassifierHead(len(vocab['syllables'])) if family == 'syllable' else None
        self.acoustic_head = nn.Linear(128, int(acoustic_dim)) if family in {'acoustic', 'components-acoustic'} else None

    def encode(self, batch: dict) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encoder.encode(batch)

    def forward(self, batch: dict) -> dict:
        hidden, mask = self.encoder.encode(batch)
        result = {'hidden': hidden, 'mask': mask}
        if self.component_heads:
            result['component_logits'] = {name: head(hidden) for name, head in self.component_heads.items()}
        if self.syllable_head is not None:
            result['syllable_logits'] = self.syllable_head(hidden)
        if self.acoustic_head is not None:
            result['acoustic_prediction'] = self.acoustic_head(hidden)
        return result

def _construct_with_streams(config: dict, vocab: dict, model_id: str, seed: int) -> TrainingModel:
    """Initialize shared weights and heads from independent seed streams."""
    model_id = canonical_model_id(model_id)
    devices = []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(int(seed))
        encoder = SharedEncoder(len(vocab['ids']['tokens']))
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(int(seed) + 1000003)
        model = TrainingModel(encoder, model_id, vocab, int(config['architecture']['acoustic_dim']))
    return model

def build_encoder(config: dict, vocab: dict, model_id: str | None=None, seed: int=0) -> TrainingModel:
    """Build a full model; the returned `.encode(batch)` is the shared export API."""
    architecture = config.get('architecture', {})
    if architecture.get('embedding_dim') != 64 or architecture.get('context_dim') != 128 or architecture.get('acoustic_dim', 0) <= 0:
        raise ValueError('experiment architecture violates encoder dimension contract')
    selected = model_id or config.get('model_id')
    if selected is None:
        raise ValueError('model_id is required to construct objective heads')
    return _construct_with_streams(config, vocab, selected, seed)

def shared_encoder_sha256(model: TrainingModel) -> str:
    import hashlib
    digest = hashlib.sha256()
    for name, tensor in sorted(model.encoder.state_dict().items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode('utf-8'))
        digest.update(str(value.dtype).encode('ascii'))
        digest.update(repr(tuple(value.shape)).encode('ascii'))
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


## Embedded `data.py`


In [ ]:
"""Torch dataset and collation for portable schema-3 experiment bundles."""
import torch
from torch.utils.data import DataLoader, Dataset

class ExperimentDataset(Dataset):

    def __init__(self, sample_ids, samples):
        self.sample_ids = list(sample_ids)
        self.samples = samples

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, index):
        return self.samples[self.sample_ids[index]]

def collate_experiment(samples):
    if not samples:
        raise ValueError('cannot collate an empty batch')
    batch_size = len(samples)
    max_sentence = max((sample['sentence_length'] for sample in samples))
    max_component = max((max(sample['component_lengths']) for sample in samples))
    ids_sequences = torch.zeros((batch_size, max_sentence, max_component), dtype=torch.long)
    component_lengths = torch.zeros((batch_size, max_sentence), dtype=torch.long)
    component_labels = torch.full((batch_size, max_sentence, 3), -100, dtype=torch.long)
    syllable_ids = torch.full((batch_size, max_sentence), -100, dtype=torch.long)
    acoustic_ids = torch.full((batch_size, max_sentence), -1, dtype=torch.long)
    sentence_lengths = torch.empty(batch_size, dtype=torch.long)
    for row_index, sample in enumerate(samples):
        n = sample['sentence_length']
        sentence_lengths[row_index] = n
        for position, (sequence, length) in enumerate(zip(sample['ids_sequences'], sample['component_lengths'])):
            if length != len(sequence) or length <= 0:
                raise ValueError('sample contains invalid IDS component length')
            ids_sequences[row_index, position, :length] = torch.as_tensor(sequence, dtype=torch.long)
            component_lengths[row_index, position] = length
        component_labels[row_index, :n] = torch.as_tensor(sample['component_labels'], dtype=torch.long)
        syllable_ids[row_index, :n] = torch.as_tensor(sample['syllable_ids'], dtype=torch.long)
        acoustic_ids[row_index, :n] = torch.as_tensor(sample['acoustic_ids'], dtype=torch.long)
    char_mask = component_lengths > 0
    return {'ids_sequences': ids_sequences, 'component_lengths': component_lengths, 'sentence_lengths': sentence_lengths, 'char_mask': char_mask, 'component_labels': component_labels, 'syllable_ids': syllable_ids, 'acoustic_ids': acoustic_ids, 'sample_ids': [sample['sample_id'] for sample in samples], 'texts': [sample['text'] for sample in samples], 'positions': [sample['positions'] for sample in samples]}

def create_loaders(experiment: dict, seed: int, batch_size: int | None=None, num_workers: int=0):
    manifest = experiment.get('manifest', {})
    if manifest.get('bundle_type') != 'experiment_dataset' or manifest.get('schema_version') != 3:
        raise ValueError('training loaders require a validated schema-3 experiment dataset')
    if type(seed) is not int or seed < 0:
        raise ValueError('loader seed must be a nonnegative integer')
    batch = batch_size or experiment['config'].get('training', {}).get('batch_size', 32)
    if type(batch) is not int or batch <= 0 or type(num_workers) is not int or (num_workers < 0):
        raise ValueError('batch_size must be positive and num_workers nonnegative')
    result = {}
    for stream, split in enumerate(('train', 'validation', 'test'), start=1):
        dataset = ExperimentDataset(experiment['splits'][split], experiment['samples'])
        result[split] = DataLoader(dataset, batch_size=batch, shuffle=split == 'train', generator=make_generator(seed, stream), num_workers=num_workers, collate_fn=collate_experiment)
    return result


## Embedded `metrics.py`


In [ ]:
"""Character-weighted losses and diagnostic metrics for training objectives."""
import torch
import torch.nn.functional as F
COSINE_EPSILON = 1e-08

def batch_objective(outputs: dict, batch: dict, acoustic_targets: torch.Tensor, model_id: str) -> dict:
    family, weight = parse_model_id(model_id)
    mask = outputs['mask']
    count = int(mask.sum().item())
    if count <= 0 or not torch.equal(mask, batch['acoustic_ids'] >= 0):
        raise ValueError('batch target masks are empty or misaligned')
    values = {'characters': count, 'objective_sum': outputs['hidden'].new_zeros(())}
    if family == 'syllable':
        target = batch['syllable_ids'][mask]
        loss = F.cross_entropy(outputs['syllable_logits'][mask], target, reduction='sum')
        values['syllable_ce_sum'] = loss
        values['objective_sum'] = values['objective_sum'] + loss
        values['syllable_correct'] = int((outputs['syllable_logits'][mask].argmax(-1) == target).sum().item())
    if family in {'components', 'components-acoustic'}:
        labels = batch['component_labels'][mask]
        ce_sums = []
        for index, name in enumerate(('initials', 'finals', 'tones')):
            logits = outputs['component_logits'][name][mask]
            target = labels[:, index]
            loss = F.cross_entropy(logits, target, reduction='sum')
            values[f'{name}_ce_sum'] = loss
            values[f'{name}_correct'] = int((logits.argmax(-1) == target).sum().item())
            ce_sums.append(loss)
        component_sum = sum(ce_sums) / 3.0
        values['component_ce_sum'] = component_sum
        values['objective_sum'] = values['objective_sum'] + component_sum
        predicted = torch.stack([outputs['component_logits'][n][mask].argmax(-1) for n in ('initials', 'finals')], dim=-1)
        valid_pairs = torch.tensor(sorted(valid_initial_final_pairs()), dtype=torch.long, device=predicted.device)
        valid = (predicted[:, None, :] == valid_pairs[None, :, :]).all(dim=-1).any(dim=-1)
        values['invalid_if_count'] = int((~valid).sum().item())
        values['joint_ift_correct'] = int((torch.stack([outputs['component_logits'][n][mask].argmax(-1) for n in ('initials', 'finals', 'tones')], dim=-1) == labels).all(dim=-1).sum().item())
    if family in {'acoustic', 'components-acoustic'}:
        target_ids = batch['acoustic_ids'][mask]
        prediction = outputs['acoustic_prediction'][mask]
        targets = acoustic_targets[target_ids]
        cosine_distance = 1.0 - F.cosine_similarity(prediction, targets, dim=-1, eps=COSINE_EPSILON)
        cosine_sum = cosine_distance.sum()
        values['cosine_sum'] = cosine_sum
        values['predicted_norm_below_epsilon'] = int((torch.linalg.vector_norm(prediction.float(), dim=-1) < COSINE_EPSILON).sum().item())
        weighted = (weight if family == 'components-acoustic' else 1.0) * cosine_sum
        values['weighted_cosine_sum'] = weighted
        values['objective_sum'] = values['objective_sum'] + weighted
    values['objective'] = values['objective_sum'] / count
    return values

def finalize_epoch(totals: dict) -> dict:
    count = int(totals.get('characters', 0))
    if count <= 0:
        raise ValueError('loader contains no valid characters')
    result = {'characters': count, 'objective': totals['objective_sum'] / count}
    for key, value in totals.items():
        if key.endswith('_sum') and key != 'objective_sum':
            result[key.removesuffix('_sum')] = value / count
    for key in ('syllable_correct', 'initials_correct', 'finals_correct', 'tones_correct', 'joint_ift_correct'):
        if key in totals:
            result[key.removesuffix('_correct') + '_accuracy'] = totals[key] / count
    if 'invalid_if_count' in totals:
        result['invalid_if_rate'] = totals['invalid_if_count'] / count
        result['invalid_if_count'] = totals['invalid_if_count']
    if 'predicted_norm_below_epsilon' in totals:
        result['predicted_norm_below_epsilon'] = totals['predicted_norm_below_epsilon']
    return result


## Embedded `training.py`


In [ ]:
"""Sequential five-seed training runs for the selected objective."""
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import shutil
import statistics
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
CODE_FILES = ('training.py', 'model.py', 'data.py', 'metrics.py', 'reproducibility.py', 'experiment.py', 'training_data.py', 'objectives.py', 'pinyin.py', 'ids.py', 'artifacts.py', 'data_contract.py')

class TrainingStepFailure(FloatingPointError):
    """A non-finite training step with JSON-safe diagnostic context."""

    def __init__(self, message, diagnostic):
        super().__init__(message)
        self.diagnostic = diagnostic

def _nonfinite_status(value):
    if math.isnan(value):
        return 'nan'
    return 'positive_infinity' if value > 0 else 'negative_infinity'

def _gradient_summary(norm_values, nonfinite_count):
    return {'mean': statistics.fmean(norm_values) if norm_values else 0.0, 'std': statistics.pstdev(norm_values) if len(norm_values) > 1 else 0.0, 'max': max(norm_values) if norm_values else 0.0, 'nonfinite_count': nonfinite_count}

def _failure_diagnostic(event, gradient_norms, nonfinite_count):
    """Retain successful work in the current epoch when a later step fails."""
    return {'nonfinite_count': int(nonfinite_count), 'gradient_norms': list(gradient_norms), 'event': event}

def _jsonable_metrics(values):
    return {key: float(value.detach().cpu()) if torch.is_tensor(value) else value for key, value in values.items()}

def _move(batch, device):
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}

def _gradient_norm(model):
    squared_norm = torch.zeros((), dtype=torch.float64, device='cpu')
    finite = True
    for parameter in model.parameters():
        if parameter.grad is None:
            continue
        grad = parameter.grad.detach()
        if not bool(torch.isfinite(grad).all()):
            finite = False
        squared_norm += grad.to(device='cpu', dtype=torch.float64).square().sum()
    value = float(torch.sqrt(squared_norm).item())
    return (value, finite and math.isfinite(value))

def _epoch(model, loader, acoustic_targets, model_id, device, *, optimizer=None, event_writer=None, epoch_number=None, step_start=0):
    training = optimizer is not None
    model.train(training)
    totals = {}
    gradient_norms = []
    nonfinite_gradients = 0
    step = step_start
    for batch in loader:
        batch = _move(batch, device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            outputs = model(batch)
            stats = batch_objective(outputs, batch, acoustic_targets, model_id)
            objective = stats['objective']
            if not bool(torch.isfinite(objective)):
                value = float(objective.detach().cpu())
                record = {'event': 'nonfinite_loss', 'epoch': epoch_number, 'step': step, 'loss': None, 'loss_status': _nonfinite_status(value)}
                if event_writer:
                    event_writer(record)
                raise TrainingStepFailure(f'non-finite loss at epoch {epoch_number}, step {step}', _failure_diagnostic(record, gradient_norms, nonfinite_gradients))
            if training:
                objective.backward()
                grad_norm, finite_grad = _gradient_norm(model)
                if not finite_grad:
                    nonfinite_gradients += 1
                    record = {'event': 'nonfinite_gradient', 'epoch': epoch_number, 'step': step, 'gradient_norm': None, 'gradient_norm_status': _nonfinite_status(grad_norm)}
                    if event_writer:
                        event_writer(record)
                    raise TrainingStepFailure(f'non-finite gradient at epoch {epoch_number}, step {step}', _failure_diagnostic(record, gradient_norms, nonfinite_gradients))
                gradient_norms.append(grad_norm)
                optimizer.step()
            else:
                grad_norm = None
        detached = _jsonable_metrics(stats)
        for key, value in detached.items():
            if key == 'objective':
                continue
            totals[key] = totals.get(key, 0) + value
        if event_writer:
            event_writer({'event': 'step', 'epoch': epoch_number, 'step': step, 'losses_per_character': {key.removesuffix('_sum'): value / stats['characters'] for key, value in detached.items() if key.endswith('_sum')}, 'objective': detached['objective'], 'gradient_norm': grad_norm})
        step += 1
    result = finalize_epoch(totals)
    result['steps'] = step - step_start
    result['gradient_norms'] = gradient_norms
    result['gradient_nonfinite_count'] = nonfinite_gradients
    return (result, step)

def _state_hash(state):
    digest = hashlib.sha256()
    for name, tensor in sorted(state.items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(value.dtype).encode())
        digest.update(repr(tuple(value.shape)).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()

def _test_summary(seed_results):
    metrics = {}
    completed = [row for row in seed_results if 'test_metrics' in row]
    keys = sorted(set.intersection(*(set(row['test_metrics']) for row in completed))) if completed else []
    for key in keys:
        values = [float(row['test_metrics'][key]) for row in completed]
        metrics[key] = {'mean': statistics.fmean(values), 'std': statistics.pstdev(values)}
    return metrics

def _write_run_manifest(run_dir, manifest):
    files = {}
    for file in sorted(run_dir.rglob('*')):
        if file.is_file() and file.name != 'manifest.json':
            files[file.relative_to(run_dir).as_posix()] = sha256_file(file)
    manifest['artifacts'] = files
    write_json(run_dir / 'manifest.json', manifest)

def train_experiment(experiment_path, *, model_id: str, runs_root=None, device: str | None=None, run_id: str | None=None, expected_manifest_sha256: str | None=None):
    """Train the selected model over the five fixed seeds, sequentially, without resume."""
    model_id = canonical_model_id(model_id)
    _, lambda_cosine = parse_model_id(model_id)
    experiment = load_experiment(experiment_path)
    experiment_root = Path(experiment['root'])
    if expected_manifest_sha256 is not None:
        actual_manifest_sha256 = sha256_file(experiment_root / 'manifest.json')
        if actual_manifest_sha256 != expected_manifest_sha256:
            raise ValueError('experiment manifest does not match the explicitly selected input pin')
    config = experiment['config']
    seeds = config['training_seeds']
    requested_device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    target_device = torch.device(requested_device)
    if target_device.type == 'cuda' and (not torch.cuda.is_available()):
        raise RuntimeError(f'selected device {requested_device!r} requires CUDA')
    experiment_id = experiment['manifest']['experiment_id']
    run_id = run_id or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ-') + uuid.uuid4().hex[:8]
    if not isinstance(run_id, str) or not run_id or Path(run_id).name != run_id or ('/' in run_id) or ('\\' in run_id):
        raise ValueError('run_id must be a safe path component')
    default_root = Path(__file__).resolve().parents[2] / 'runs'
    output_root = Path(runs_root).resolve() if runs_root else default_root
    run_dir = output_root / experiment_id / model_id / run_id
    if run_dir.exists():
        raise FileExistsError(f'run already exists: {run_dir}')
    run_dir.parent.mkdir(parents=True, exist_ok=True)
    run_dir.mkdir(parents=False, exist_ok=False)
    manifest = {'schema_version': 1, 'run_type': 'training_run', 'status': 'running', 'run_id': run_id, 'experiment_id': experiment_id, 'model_id': model_id, 'lambda_cosine': lambda_cosine, 'device': str(target_device), 'seeds': seeds, 'selection': 'minimum per-seed validation objective; strict improvement keeps earliest tie', 'experiment_hashes': {name: sha256_file(experiment_root / name) for name in ('manifest.json', 'config.json', 'vocab.json', 'samples.jsonl', 'splits.json', 'acoustic_targets.npy')}, 'config': config, 'code_sha256': {name: sha256_file(Path(__file__).with_name(name)) for name in CODE_FILES}, 'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'cuda_available': torch.cuda.is_available(), 'cuda_version': torch.version.cuda}, 'seed_results': [], 'summary': None, 'artifacts': {}}
    write_json(run_dir / 'manifest.json', manifest)
    seed_results = []
    statuses = {str(seed): 'pending' for seed in seeds}
    global_step = 0
    active_seed = None
    try:
        for seed in seeds:
            active_seed = seed
            norm_values = []
            nonfinite_count = 0
            seed_dir = run_dir / f'seed-{seed}'
            seed_dir.mkdir()
            statuses[str(seed)] = 'running'
            event_path = seed_dir / 'events.jsonl'

            def log_event(event):
                with event_path.open('a', encoding='utf-8', newline='\n') as handle:
                    handle.write(json.dumps(event, sort_keys=True, allow_nan=False) + '\n')
            seed_everything(seed)
            model = build_encoder(config, experiment['vocab'], model_id=model_id, seed=seed).to(target_device)
            initial_state = {key: value.detach().cpu().clone() for key, value in model.encoder.state_dict().items()}
            shared_hash = _state_hash(initial_state)
            torch.save({'state_dict': initial_state, 'seed': seed, 'sha256': shared_hash}, seed_dir / 'shared_encoder.pt')
            loaders = create_loaders(experiment, seed, batch_size=32, num_workers=0)
            acoustic = torch.as_tensor(experiment['acoustic_targets'], dtype=torch.float32, device=target_device)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0)
            best = math.inf
            best_epoch = None
            best_validation = None
            epoch_history = []
            for epoch_num in range(1, 21):
                train_metrics, global_step = _epoch(model, loaders['train'], acoustic, model_id, target_device, optimizer=optimizer, event_writer=log_event, epoch_number=epoch_num, step_start=global_step)
                norm_values.extend(train_metrics.pop('gradient_norms'))
                nonfinite_count += train_metrics.pop('gradient_nonfinite_count')
                validation_metrics, _ = _epoch(model, loaders['validation'], acoustic, model_id, target_device, event_writer=log_event, epoch_number=epoch_num)
                val_objective = float(validation_metrics['objective'])
                epoch_history.append({'epoch': epoch_num, 'train': train_metrics, 'validation': validation_metrics})
                if val_objective < best:
                    best, best_epoch, best_validation = (val_objective, epoch_num, validation_metrics)
                    torch.save({'state_dict': model.state_dict(), 'model_id': model_id, 'lambda_cosine': lambda_cosine, 'seed': seed, 'best_epoch': best_epoch, 'validation_objective': best, 'shared_encoder_sha256': shared_hash, 'experiment_hashes': manifest['experiment_hashes'], 'config_sha256': sha256_file(experiment_root / 'config.json'), 'vocab_sha256': sha256_file(experiment_root / 'vocab.json')}, seed_dir / 'best.pt')
                torch.save({'state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'seed': seed, 'epoch': epoch_num, 'rng_state': rng_snapshot(loaders['train'].generator)}, seed_dir / 'last.pt')
                write_json(seed_dir / 'history.json', {'epochs': epoch_history})
                log_event({'event': 'epoch', 'epoch': epoch_num, 'train': train_metrics, 'validation': validation_metrics, 'best_epoch': best_epoch, 'best_validation_objective': best})
            best_checkpoint = torch.load(seed_dir / 'best.pt', map_location=target_device, weights_only=True)
            model.load_state_dict(best_checkpoint['state_dict'], strict=True)
            test_metrics, _ = _epoch(model, loaders['test'], acoustic, model_id, target_device, event_writer=log_event, epoch_number='test')
            test_metrics.pop('gradient_norms')
            test_metrics.pop('gradient_nonfinite_count')
            grad_summary = _gradient_summary(norm_values, nonfinite_count)
            result = {'seed': seed, 'status': 'complete', 'best_epoch': best_epoch, 'best_validation_objective': best, 'best_validation_metrics': best_validation, 'test_metrics': test_metrics, 'gradient_norm': grad_summary, 'shared_encoder_sha256': shared_hash}
            seed_results.append(result)
            statuses[str(seed)] = 'complete'
            write_json(seed_dir / 'summary.json', result)
            manifest['seed_results'] = seed_results
            manifest['seed_status'] = statuses
            write_json(run_dir / 'summary.json', {'status': 'running', 'completed_seeds': len(seed_results), 'seed_results': seed_results, 'test_metrics': _test_summary(seed_results)})
            _write_run_manifest(run_dir, manifest)
            active_seed = None
        gradient = [row['gradient_norm'] for row in seed_results]
        run_summary = {'seeds_completed': len(seed_results), 'test_metrics': _test_summary(seed_results), 'gradient_norm': {'mean_of_seed_means': statistics.fmean((x['mean'] for x in gradient)), 'max_of_seed_maxima': max((x['max'] for x in gradient)), 'nonfinite_count': sum((x['nonfinite_count'] for x in gradient))}}
        manifest.update({'status': 'complete', 'seed_status': statuses, 'seed_results': seed_results, 'summary': run_summary})
        write_json(run_dir / 'summary.json', {'status': 'complete', **run_summary, 'seed_results': seed_results})
        _write_run_manifest(run_dir, manifest)
        return {'run_dir': run_dir, 'manifest': manifest, 'summary': run_summary}
    except Exception as exc:
        failure_diagnostic = getattr(exc, 'diagnostic', None)
        if active_seed is not None and statuses.get(str(active_seed)) == 'running':
            prior_norms = list(locals().get('norm_values', []))
            prior_nonfinite = int(locals().get('nonfinite_count', 0))
            if failure_diagnostic:
                prior_norms.extend(failure_diagnostic.get('gradient_norms', []))
                prior_nonfinite += int(failure_diagnostic.get('nonfinite_count', 0))
            failed_result = {'seed': active_seed, 'status': 'failed', 'gradient_norm': _gradient_summary(prior_norms, prior_nonfinite)}
            if failure_diagnostic:
                failed_result['failure_event'] = failure_diagnostic['event']
            seed_results.append(failed_result)
            statuses[str(active_seed)] = 'failed'
            failed_dir = run_dir / f'seed-{active_seed}'
            if failed_dir.is_dir():
                write_json(failed_dir / 'summary.json', failed_result)
        statuses.update({key: 'failed' if value == 'running' else value for key, value in statuses.items()})
        status = 'partial' if seed_results else 'failed'
        failure = {'exception': type(exc).__name__, 'message': str(exc), 'completed_seeds': sum((x['status'] == 'complete' for x in seed_results)), 'seed_status': statuses, 'global_step': global_step, 'diagnostic': failure_diagnostic}
        write_json(run_dir / 'failure.json', failure)
        manifest.update({'status': status, 'seed_status': statuses, 'seed_results': seed_results, 'failure': failure})
        write_json(run_dir / 'summary.json', {'status': status, 'seed_results': seed_results, 'test_metrics': _test_summary(seed_results), 'gradient_norm': {'mean_of_seed_means': statistics.fmean((x['gradient_norm']['mean'] for x in seed_results)), 'max_of_seed_maxima': max((x['gradient_norm']['max'] for x in seed_results)), 'nonfinite_count': sum((x['gradient_norm']['nonfinite_count'] for x in seed_results))}})
        _write_run_manifest(run_dir, manifest)
        raise


## Embedded `evaluation/dimsim.py`


In [ ]:
"""Pinned public DIMSIM adapter used by evaluation-bundle preparation."""
import hashlib
from importlib import metadata
from pathlib import Path
DIMSIM_VERSION = '0.2.2'
PYPINYIN_VERSION = '0.55.0'

class DimsimDistance:
    """Validated callable around the public ``dimsim.get_distance`` API."""

    def __init__(self) -> None:
        versions = {'dimsim': metadata.version('dimsim'), 'pypinyin': metadata.version('pypinyin')}
        expected = {'dimsim': DIMSIM_VERSION, 'pypinyin': PYPINYIN_VERSION}
        if versions != expected:
            raise RuntimeError(f'DIMSIM dependency versions {versions} do not match pinned {expected}')
        import dimsim
        if not callable(getattr(dimsim, 'get_distance', None)):
            raise RuntimeError('dimsim.get_distance public API is unavailable')
        self._get_distance = dimsim.get_distance
        package_root = Path(dimsim.__file__).resolve().parent
        source_hashes = {}
        for path in sorted(package_root.rglob('*')):
            if path.is_file() and (path.suffix == '.py' or path.suffix.startswith('.pickle')):
                source_hashes[str(path.relative_to(package_root)).replace('\\', '/')] = hashlib.sha256(path.read_bytes()).hexdigest()
        self.provenance = {'metric_name': 'dimsim', 'metric_version': DIMSIM_VERSION, 'dependencies': versions, 'api': 'dimsim.get_distance([left], [right], pinyin=True)', 'canonical_weights_modified': False, 'package_file_sha256': source_hashes}

    def __call__(self, left: str, right: str) -> float:
        return float(self._get_distance([left], [right], pinyin=True))


## Embedded `evaluation/sampling.py`


In [ ]:
"""Deterministic sampling of unique unordered pairs without quadratic materialization."""
import random
from collections.abc import Iterator, Sequence

def pair_count(item_count: int) -> int:
    """Return the number of unique unordered pairs of ``item_count`` objects."""
    if item_count < 0:
        raise ValueError('item_count must be non-negative')
    return item_count * (item_count - 1) // 2

def pair_from_rank(item_count: int, rank: int) -> tuple[int, int]:
    """Map a rank in lexicographic upper-triangle order to ``(left, right)``."""
    total = pair_count(item_count)
    if rank < 0 or rank >= total:
        raise IndexError(f'pair rank {rank} outside [0, {total})')
    low, high = (0, item_count - 1)
    while low + 1 < high:
        middle = (low + high) // 2
        before = middle * (2 * item_count - middle - 1) // 2
        if before <= rank:
            low = middle
        else:
            high = middle
    left = low
    before = left * (2 * item_count - left - 1) // 2
    right = left + 1 + (rank - before)
    return (left, right)

def iter_all_pairs(item_count: int, *, max_pairs: int) -> Iterator[tuple[int, int]]:
    """Yield all unique pairs, rejecting an unexpectedly large expansion."""
    total = pair_count(item_count)
    if max_pairs < 0:
        raise ValueError('max_pairs must be non-negative')
    if total > max_pairs:
        raise ValueError(f'all-pairs plan has {total} pairs, above cap {max_pairs}')
    for left in range(item_count):
        for right in range(left + 1, item_count):
            yield (left, right)

def sample_pair_indices(item_count: int, sample_count: int, seed: int) -> list[tuple[int, int]]:
    """Uniformly sample unique unordered pairs without constructing all pairs."""
    total = pair_count(item_count)
    if sample_count < 0:
        raise ValueError('sample_count must be non-negative')
    if sample_count > total:
        raise ValueError(f'requested {sample_count} pairs from population of {total}')
    ranks = random.Random(seed).sample(range(total), sample_count)
    return [pair_from_rank(item_count, rank) for rank in sorted(ranks)]

def make_pair_plan(item_ids: Sequence[str], *, sample_count: int | None, seed: int, max_all_pairs: int) -> list[tuple[str, str]]:
    """Return canonical ID pairs from an already stably ordered collection."""
    if len(item_ids) != len(set(item_ids)):
        raise ValueError('item_ids must be unique')
    if list(item_ids) != sorted(item_ids):
        raise ValueError('item_ids must be stably sorted')
    if sample_count is None:
        indices = iter_all_pairs(len(item_ids), max_pairs=max_all_pairs)
    else:
        indices = sample_pair_indices(len(item_ids), sample_count, seed)
    return [(item_ids[left], item_ids[right]) for left, right in indices]


## Embedded `evaluation/bundle.py`


In [ ]:
"""Create and validate immutable, model-independent evaluation bundles.

The on-disk format intentionally uses JSON/JSONL so the preparation stage has
no scientific-runtime dependency. Embeddings are produced by later stages.
"""
import hashlib
import json
import random
import re
import shutil
from collections import defaultdict
from collections.abc import Callable, Iterable
from pathlib import Path
from typing import Any
BUNDLE_SCHEMA_VERSION = 2
PAIRING_POLICY = 'nonzero_nonzero'
_EVALUATION_ID_PATTERN = re.compile('^[A-Za-z0-9](?:[A-Za-z0-9._-]*[A-Za-z0-9])?$')
_WINDOWS_RESERVED_NAME_PATTERN = re.compile('^(?:CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9])(?:\\..*)?$', re.IGNORECASE)
DistanceFunction = Callable[[str, str], float]

def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def _write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')

def _write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.write_text(''.join((json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in rows)), encoding='utf-8')

def _read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))

def _read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]

def _safe_id(value: str, field: str) -> str:
    if not isinstance(value, str) or not value:
        raise ValueError(f'{field} must be a non-empty string')
    return value

def _validate_evaluation_id(value: Any) -> str:
    if not isinstance(value, str) or _EVALUATION_ID_PATTERN.fullmatch(value) is None or '..' in value or ('/' in value) or ('\\' in value) or (':' in value) or value.endswith(('.', ' ')) or Path(value).is_absolute() or (_WINDOWS_RESERVED_NAME_PATTERN.fullmatch(value) is not None):
        raise ValueError("evaluation_id must be a safe single-directory name using ASCII letters, digits, '.', '_', or '-'")
    return value

def _occurrence_id(sample_id: str, position: int) -> str:
    return f'{sample_id}:{position}'

def _group_id(kind: str, values: tuple[str, ...]) -> str:
    payload = json.dumps([kind, *values], ensure_ascii=False, separators=(',', ':'))
    return f"{kind}:{hashlib.sha256(payload.encode('utf-8')).hexdigest()}"

def _load_source_experiment(root: Path, split: str, sentence_limit: int | None, sentence_seed: int) -> tuple[list[dict], dict, dict, dict]:
    files = {name: root / name for name in ('manifest.json', 'samples.jsonl', 'splits.json', 'vocab.json', 'config.json')}
    missing = [str(path) for path in files.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(missing)
    manifest = _read_json(files['manifest.json'])
    if manifest.get('pinyin_contract') != PINYIN_CONTRACT_VERSION or manifest.get('parser_provenance') != PARSER_PROVENANCE:
        raise ValueError('unsupported source experiment Pinyin contract')
    artifacts = manifest.get('artifacts', {})
    for name in ('samples.jsonl', 'splits.json', 'vocab.json', 'config.json'):
        expected = artifacts.get(name)
        if not isinstance(expected, str) or _sha256(files[name]) != expected:
            raise ValueError(f'source experiment artifact checksum mismatch: {name}')
    splits, vocab = (_read_json(files['splits.json']), _read_json(files['vocab.json']))
    if split not in splits or not isinstance(splits[split], list) or (not splits[split]):
        raise ValueError(f'unknown experiment split: {split}')
    if sentence_limit is not None and (isinstance(sentence_limit, bool) or not isinstance(sentence_limit, int) or sentence_limit <= 0):
        raise ValueError('sentence_limit must be positive or None')
    if any((not isinstance(values, list) for values in splits.values())):
        raise ValueError('source experiment splits must contain ID lists')
    all_split_ids = [sample_id for values in splits.values() for sample_id in values]
    if any((not isinstance(value, str) or not value for value in all_split_ids)) or len(all_split_ids) != len(set(all_split_ids)):
        raise ValueError('source experiment splits are not disjoint unique ID lists')
    selected_ids = sorted(splits[split])
    if sentence_limit is not None:
        if sentence_limit > len(selected_ids):
            raise ValueError(f'sentence_limit {sentence_limit} exceeds split size {len(selected_ids)}')
        selected_ids = sorted(random.Random(sentence_seed).sample(selected_ids, sentence_limit))
    sample_rows = _read_jsonl(files['samples.jsonl'])
    samples = {row['sample_id']: row for row in sample_rows}
    if len(samples) != len(sample_rows):
        raise ValueError('duplicate sample IDs in source experiment')
    try:
        selected = [samples[sample_id] for sample_id in selected_ids]
    except KeyError as error:
        raise ValueError(f'split references missing sample: {error.args[0]}') from error
    hashes = {name: _sha256(files[name]) for name in ('manifest.json', 'samples.jsonl', 'splits.json', 'vocab.json', 'config.json')}
    return (selected, vocab, manifest, hashes)

def _build_occurrences(samples: list[dict], vocab: dict) -> list[dict]:
    acoustic_tokens = vocab.get('acoustic_tokens')
    labels = vocab.get('labels', {})
    if not isinstance(acoustic_tokens, list) or any((not isinstance(x, str) for x in acoustic_tokens)):
        raise ValueError('invalid acoustic token vocabulary')
    component_names = [labels.get(name) for name in ('initials', 'finals', 'tones')]
    if any((not isinstance(values, list) or any((not isinstance(value, str) for value in values)) for values in component_names)):
        raise ValueError('invalid component label vocabulary')
    rows = []
    for sample in sorted(samples, key=lambda row: row['sample_id']):
        sample_id, text = (_safe_id(sample.get('sample_id'), 'sample_id'), sample.get('text'))
        acoustic_ids, components = (sample.get('acoustic_ids'), sample.get('component_labels'))
        if not isinstance(text, str) or not len(text) == len(acoustic_ids or []) == len(components or []):
            raise ValueError(f'source sample shape mismatch: {sample_id}')
        for position, character in enumerate(text):
            acoustic_id = acoustic_ids[position]
            component_ids = components[position]
            if not isinstance(acoustic_id, int) or acoustic_id < 0 or acoustic_id >= len(acoustic_tokens):
                raise ValueError(f'invalid acoustic ID in sample {sample_id}')
            if not isinstance(component_ids, list) or len(component_ids) != 3:
                raise ValueError(f'invalid component labels in sample {sample_id}')
            if any((isinstance(value, bool) or not isinstance(value, int) or value < 0 or (value >= len(component_names[i])) for i, value in enumerate(component_ids))):
                raise ValueError(f'component ID outside vocabulary in sample {sample_id}')
            try:
                initial, final, tone = (component_names[i][value] for i, value in enumerate(component_ids))
            except (IndexError, TypeError) as error:
                raise ValueError(f'component ID outside vocabulary in sample {sample_id}') from error
            pinyin = acoustic_tokens[acoustic_id]
            parsed = parse_components(validate_canonical_label(pinyin))
            if (initial, final, tone) != (parsed.initial, parsed.final, parsed.tone):
                raise ValueError(f'component labels disagree with acoustic label in sample {sample_id}')
            rows.append({'occurrence_id': _occurrence_id(sample_id, position), 'sample_id': sample_id, 'character_position': position, 'character': character, 'pinyin': pinyin, 'initial': initial, 'final': final, 'tone': tone})
    return sorted(rows, key=lambda row: row['occurrence_id'])

def _build_groups(occurrences: list[dict]) -> tuple[list[dict], dict[str, dict]]:
    members: dict[str, list[str]] = defaultdict(list)
    metadata: dict[str, dict] = {}
    for row in occurrences:
        definitions = (('character_pinyin', (row['character'], row['pinyin'])), ('syllable', (row['pinyin'],)))
        for kind, keys in definitions:
            group_id = _group_id(kind, keys)
            members[group_id].append(row['occurrence_id'])
            metadata[group_id] = {'group_id': group_id, 'group_kind': kind, 'keys': list(keys), 'pinyin': row['pinyin'], 'initial': row['initial'], 'final': row['final'], 'tone': row['tone']}
    groups = []
    for group_id in sorted(members):
        groups.append({**metadata[group_id], 'occurrence_ids': sorted(members[group_id]), 'occurrence_count': len(members[group_id])})
    return (groups, {row['group_id']: row for row in groups})

def _pair_rows(ids: list[str], metadata: dict[str, dict], *, sample_count: int | None, seed: int, max_all_pairs: int, distance_fn: DistanceFunction | None) -> list[dict]:
    rows = []
    for left_id, right_id in make_pair_plan(ids, sample_count=sample_count, seed=seed, max_all_pairs=max_all_pairs):
        left, right = (metadata[left_id], metadata[right_id])
        status, distance = ('not_computed', None)
        if distance_fn is not None:
            try:
                distance = float(distance_fn(left['pinyin'], right['pinyin']))
                if distance != distance or distance in (float('inf'), float('-inf')):
                    raise ValueError('distance is not finite')
                status = 'ok'
            except Exception as error:
                status = f'invalid:{type(error).__name__}'
                distance = None
        equal_initial = left['initial'] == right['initial']
        equal_final = left['final'] == right['final']
        equal_tone = left['tone'] == right['tone']
        pair_id = hashlib.sha256(json.dumps([left_id, right_id], ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
        rows.append({'pair_id': pair_id, 'left_id': left_id, 'right_id': right_id, 'left_pinyin': left['pinyin'], 'right_pinyin': right['pinyin'], 'dimsim_distance': distance, 'distance_status': status, 'equal_initial': equal_initial, 'equal_final': equal_final, 'equal_tone': equal_tone, 'overlap_mask': f'{int(equal_initial)}{int(equal_final)}{int(equal_tone)}', 'left_zero_initial': left['initial'] == '', 'right_zero_initial': right['initial'] == '', 'no_zero_initial_diagnostic': left['initial'] != '' and right['initial'] != ''})
    return rows

def prepare_evaluation_bundle(evaluation_id: str, *, experiment_path: str | Path, output_root: str | Path, split: str='test', sentence_limit: int | None=None, contextual_pair_count: int=12000, sentence_seed: int=42, sampling_seed: int=42, character_pair_count: int=12000, syllable_pair_count: int | None=None, max_all_pairs: int=1000000, distance_fn: DistanceFunction | None=None, metric_provenance: dict[str, Any] | None=None, code_provenance: dict[str, str] | None=None) -> Path:
    """Create an immutable evaluation bundle and return its path."""
    evaluation_id = _validate_evaluation_id(evaluation_id)
    numeric = {'contextual_pair_count': contextual_pair_count, 'sentence_seed': sentence_seed, 'sampling_seed': sampling_seed, 'character_pair_count': character_pair_count, 'max_all_pairs': max_all_pairs}
    if any((isinstance(value, bool) or not isinstance(value, int) for value in numeric.values())):
        raise TypeError('pair counts, caps, and seeds must be integers')
    if contextual_pair_count < 0 or character_pair_count < 0 or max_all_pairs <= 0:
        raise ValueError('pair counts must be non-negative and max_all_pairs positive')
    if syllable_pair_count is not None and (isinstance(syllable_pair_count, bool) or not isinstance(syllable_pair_count, int) or syllable_pair_count < 0):
        raise ValueError('syllable_pair_count must be a non-negative integer or None')
    if distance_fn is None or not callable(distance_fn):
        raise ValueError('an explicit validated distance provider is required')
    if not isinstance(metric_provenance, dict) or not isinstance(metric_provenance.get('metric_name'), str):
        raise ValueError('metric_provenance with an explicit metric_name is required')
    metric_name = metric_provenance['metric_name']
    if metric_name == 'dimsim':
        if not isinstance(distance_fn, DimsimDistance) or metric_provenance != distance_fn.provenance:
            raise ValueError('DIMSIM bundles require the validated pinned DimsimDistance provider and its provenance')
    elif metric_name != 'synthetic':
        raise ValueError('only the pinned DIMSIM provider or an explicitly labelled synthetic test provider is supported')
    if not isinstance(code_provenance, dict) or not code_provenance or any((not isinstance(key, str) or not isinstance(value, str) or len(value) != 64 or any((character not in '0123456789abcdef' for character in value)) for key, value in code_provenance.items())):
        raise ValueError('explicit non-empty preparation code_provenance hashes are required')
    experiment_root, root = (Path(experiment_path).resolve(), Path(output_root).resolve())
    target = (root / evaluation_id).resolve()
    if target.parent != root:
        raise ValueError('evaluation path escapes output_root')
    if target == experiment_root or experiment_root in target.parents:
        raise ValueError('evaluation output must not be inside the source experiment')
    if target.exists():
        raise FileExistsError(f'evaluation bundle already exists: {target}')
    samples, vocab, source_manifest, source_hashes = _load_source_experiment(experiment_root, split, sentence_limit, sentence_seed)
    occurrences = _build_occurrences(samples, vocab)
    groups, group_by_id = _build_groups(occurrences)
    occurrence_by_id = {row['occurrence_id']: row for row in occurrences}
    eligible_ids = {'contextual': sorted((row['occurrence_id'] for row in occurrences if isinstance(row['initial'], str) and row['initial'])), 'character_pinyin': sorted((row['group_id'] for row in groups if row['group_kind'] == 'character_pinyin' and isinstance(row['initial'], str) and row['initial'])), 'syllable': sorted((row['group_id'] for row in groups if row['group_kind'] == 'syllable' and isinstance(row['initial'], str) and row['initial']))}
    contextual = _pair_rows(eligible_ids['contextual'], occurrence_by_id, sample_count=contextual_pair_count, seed=sampling_seed, max_all_pairs=max_all_pairs, distance_fn=distance_fn)
    aggregate_plans = {}
    pair_counts = {'character_pinyin': character_pair_count, 'syllable': syllable_pair_count}
    for offset, kind in enumerate(('character_pinyin', 'syllable'), start=1):
        aggregate_plans[kind] = _pair_rows(eligible_ids[kind], group_by_id, sample_count=pair_counts[kind], seed=sampling_seed + offset, max_all_pairs=max_all_pairs, distance_fn=distance_fn)
    target.mkdir(parents=True)
    try:
        (target / 'pairs').mkdir()
        selected_sentences = [{'sample_id': row['sample_id'], 'text': row['text']} for row in sorted(samples, key=lambda x: x['sample_id'])]
        _write_jsonl(target / 'sentences.jsonl', selected_sentences)
        _write_jsonl(target / 'occurrences.jsonl', occurrences)
        _write_jsonl(target / 'groups.jsonl', groups)
        _write_jsonl(target / 'pairs/contextual.jsonl', contextual)
        for kind, rows in aggregate_plans.items():
            _write_jsonl(target / f'pairs/{kind}.jsonl', rows)
        object_counts = {'contextual': len(occurrences), 'character_pinyin': sum((row['group_kind'] == 'character_pinyin' for row in groups)), 'syllable': sum((row['group_kind'] == 'syllable' for row in groups))}
        all_plans = {'contextual': contextual, **aggregate_plans}
        coverage = {'sentences': len(samples), 'occurrences': len(occurrences), 'groups': {kind: object_counts[kind] for kind in ('character_pinyin', 'syllable')}, 'levels': {}}
        for level, rows in all_plans.items():
            statuses: dict[str, int] = {}
            for row in rows:
                statuses[row['distance_status']] = statuses.get(row['distance_status'], 0) + 1
            original_count = object_counts[level]
            eligible_count = len(eligible_ids[level])
            original_pairs, eligible_pairs = (pair_count(original_count), pair_count(eligible_count))
            coverage['levels'][level] = {'objects': {'original': original_count, 'eligible': eligible_count, 'excluded': original_count - eligible_count, 'excluded_by_reason': {'zero_initial': original_count - eligible_count}}, 'unordered_pair_population': {'original': original_pairs, 'eligible': eligible_pairs, 'excluded': original_pairs - eligible_pairs, 'excluded_by_reason': {'zero_initial': original_pairs - eligible_pairs}}, 'selection_count': len(rows), 'distance_status_counts': dict(sorted(statuses.items()))}
        _write_json(target / 'coverage.json', coverage)
        config = {'schema_version': BUNDLE_SCHEMA_VERSION, 'evaluation_id': evaluation_id, 'split': split, 'sentence_limit': sentence_limit, 'contextual_pair_count': contextual_pair_count, 'sentence_seed': sentence_seed, 'sampling_seed': sampling_seed, 'character_pair_count': character_pair_count, 'syllable_pair_count': syllable_pair_count, 'max_all_pairs': max_all_pairs, 'pairing_policy': PAIRING_POLICY}
        _write_json(target / 'config.json', config)
        artifacts = {str(path.relative_to(target)).replace('\\', '/'): _sha256(path) for path in target.rglob('*') if path.is_file()}
        manifest = {'schema_version': BUNDLE_SCHEMA_VERSION, 'evaluation_id': evaluation_id, 'pinyin_contract': PINYIN_CONTRACT_VERSION, 'parser_provenance': PARSER_PROVENANCE, 'pairing_policy': PAIRING_POLICY, 'source_experiment': {'path': str(experiment_root), 'experiment_id': source_manifest.get('experiment_id'), 'hashes': source_hashes}, 'metric': metric_provenance, 'code_provenance': code_provenance, 'counts': {'sentences': len(samples), 'occurrences': len(occurrences), 'character_pinyin_groups': sum((row['group_kind'] == 'character_pinyin' for row in groups)), 'syllable_groups': sum((row['group_kind'] == 'syllable' for row in groups)), 'pairs': {'contextual': len(contextual), **{kind: len(rows) for kind, rows in aggregate_plans.items()}}}, 'artifacts': artifacts}
        _write_json(target / 'manifest.json', manifest)
    except Exception:
        shutil.rmtree(target)
        raise
    return target

def load_evaluation_bundle(path: str | Path) -> dict[str, Any]:
    """Load a bundle after verifying every recorded artifact checksum."""
    root = Path(path).resolve()
    manifest = _read_json(root / 'manifest.json')
    if manifest.get('schema_version') != BUNDLE_SCHEMA_VERSION:
        raise ValueError('unsupported evaluation bundle schema')
    if manifest.get('pinyin_contract') != PINYIN_CONTRACT_VERSION or manifest.get('parser_provenance') != PARSER_PROVENANCE:
        raise ValueError('unsupported evaluation Pinyin contract')
    config = _read_json(root / 'config.json')
    if config.get('schema_version') != BUNDLE_SCHEMA_VERSION:
        raise ValueError('unsupported evaluation config schema')
    if manifest.get('pairing_policy') != PAIRING_POLICY or config.get('pairing_policy') != PAIRING_POLICY:
        raise ValueError('unsupported evaluation pairing policy')
    manifest_evaluation_id = _validate_evaluation_id(manifest.get('evaluation_id'))
    config_evaluation_id = _validate_evaluation_id(config.get('evaluation_id'))
    if manifest_evaluation_id != config_evaluation_id:
        raise ValueError('invalid or inconsistent evaluation_id')
    required = {'config.json', 'coverage.json', 'sentences.jsonl', 'occurrences.jsonl', 'groups.jsonl', 'pairs/contextual.jsonl', 'pairs/character_pinyin.jsonl', 'pairs/syllable.jsonl'}
    artifacts = manifest.get('artifacts', {})
    if not required.issubset(artifacts):
        raise ValueError('evaluation manifest is missing required artifacts')
    for relative, expected in artifacts.items():
        artifact = (root / relative).resolve()
        if root not in artifact.parents or not artifact.is_file() or _sha256(artifact) != expected:
            raise ValueError(f'evaluation artifact checksum mismatch: {relative}')
    coverage = _read_json(root / 'coverage.json')
    occurrences, groups = (_read_jsonl(root / 'occurrences.jsonl'), _read_jsonl(root / 'groups.jsonl'))
    occurrence_ids = [row.get('occurrence_id') for row in occurrences]
    group_ids = [row.get('group_id') for row in groups]
    if len(occurrence_ids) != len(set(occurrence_ids)) or any((not isinstance(value, str) for value in occurrence_ids)):
        raise ValueError('evaluation occurrence IDs are invalid or duplicated')
    if len(group_ids) != len(set(group_ids)) or any((not isinstance(value, str) for value in group_ids)):
        raise ValueError('evaluation group IDs are invalid or duplicated')
    if any((not isinstance(row.get('initial'), str) for row in occurrences + groups)):
        raise ValueError('evaluation initial metadata must be strings')
    for row in occurrences + groups:
        label = validate_canonical_label(row.get('pinyin'))
        parsed = parse_components(label)
        if tuple((row.get(key) for key in ('initial', 'final', 'tone'))) != (parsed.initial, parsed.final, parsed.tone):
            raise ValueError('stored evaluation components disagree with canonical Pinyin label')
    try:
        _, expected_groups = _build_groups(occurrences)
    except (KeyError, TypeError) as error:
        raise ValueError('invalid occurrence group metadata') from error
    if {row['group_id']: row for row in groups} != expected_groups:
        raise ValueError('stored evaluation group metadata is inconsistent with occurrences')
    pairs = {kind: _read_jsonl(root / f'pairs/{kind}.jsonl') for kind in ('contextual', 'character_pinyin', 'syllable')}
    occurrence_id_set = set(occurrence_ids)
    valid_ids = {'contextual': occurrence_id_set, 'character_pinyin': {row['group_id'] for row in groups if row.get('group_kind') == 'character_pinyin'}, 'syllable': {row['group_id'] for row in groups if row.get('group_kind') == 'syllable'}}
    metadata = {'contextual': {row['occurrence_id']: row for row in occurrences}, 'character_pinyin': {row['group_id']: row for row in groups if row.get('group_kind') == 'character_pinyin'}, 'syllable': {row['group_id']: row for row in groups if row.get('group_kind') == 'syllable'}}
    for group in groups:
        members = group.get('occurrence_ids')
        if not isinstance(members, list) or not members or members != sorted(members) or (len(members) != len(set(members))) or (group.get('occurrence_count') != len(members)) or any((member not in occurrence_id_set for member in members)):
            raise ValueError(f"invalid group membership: {group.get('group_id')}")
    for kind, rows in pairs.items():
        pair_ids = [row.get('pair_id') for row in rows]
        if len(pair_ids) != len(set(pair_ids)) or any((not isinstance(value, str) for value in pair_ids)):
            raise ValueError(f'{kind} pair IDs are invalid or duplicated')
        if any((row.get('left_id') not in valid_ids[kind] or row.get('right_id') not in valid_ids[kind] for row in rows)):
            raise ValueError(f'{kind} pair references an unknown object')
        endpoints = [(row['left_id'], row['right_id']) for row in rows]
        if len(endpoints) != len(set(endpoints)) or any((left >= right for left, right in endpoints)):
            raise ValueError(f'{kind} pairs are duplicated or non-canonical')
        for row in rows:
            expected = hashlib.sha256(json.dumps([row['left_id'], row['right_id']], ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
            if row['pair_id'] != expected:
                raise ValueError(f'{kind} pair ID does not match endpoints')
            left = metadata[kind][row['left_id']]
            right = metadata[kind][row['right_id']]
            left_zero = metadata[kind][row['left_id']].get('initial') == ''
            right_zero = metadata[kind][row['right_id']].get('initial') == ''
            if left_zero or right_zero:
                raise ValueError(f'{kind} pair violates {PAIRING_POLICY} policy')
            if row.get('left_zero_initial') is not False or row.get('right_zero_initial') is not False or row.get('no_zero_initial_diagnostic') is not True:
                raise ValueError(f'{kind} pair has inconsistent zero-initial flags')
            if row.get('left_pinyin') != left['pinyin'] or row.get('right_pinyin') != right['pinyin']:
                raise ValueError(f'{kind} pair Pinyin labels disagree with endpoints')
            equal = [left[key] == right[key] for key in ('initial', 'final', 'tone')]
            if any((row.get('equal_' + key) is not value for key, value in zip(('initial', 'final', 'tone'), equal))) or row.get('overlap_mask') != ''.join((str(int(value)) for value in equal)):
                raise ValueError(f'{kind} pair component metadata disagrees with endpoints')
        original_count = len(valid_ids[kind])
        eligible_count = sum((isinstance(metadata[kind][object_id].get('initial'), str) and bool(metadata[kind][object_id]['initial']) for object_id in valid_ids[kind]))
        original_pairs, eligible_pairs = (pair_count(original_count), pair_count(eligible_count))
        statuses: dict[str, int] = {}
        for row in rows:
            status = row.get('distance_status')
            statuses[status] = statuses.get(status, 0) + 1
        expected_coverage = {'objects': {'original': original_count, 'eligible': eligible_count, 'excluded': original_count - eligible_count, 'excluded_by_reason': {'zero_initial': original_count - eligible_count}}, 'unordered_pair_population': {'original': original_pairs, 'eligible': eligible_pairs, 'excluded': original_pairs - eligible_pairs, 'excluded_by_reason': {'zero_initial': original_pairs - eligible_pairs}}, 'selection_count': len(rows), 'distance_status_counts': dict(sorted(statuses.items()))}
        if coverage.get('levels', {}).get(kind) != expected_coverage:
            raise ValueError(f'{kind} coverage metadata is inconsistent')
    expected_counts = {'sentences': len(_read_jsonl(root / 'sentences.jsonl')), 'occurrences': len(occurrences), 'character_pinyin_groups': len(valid_ids['character_pinyin']), 'syllable_groups': len(valid_ids['syllable']), 'pairs': {kind: len(rows) for kind, rows in pairs.items()}}
    if manifest.get('counts') != expected_counts:
        raise ValueError('evaluation manifest counts are inconsistent')
    return {'root': root, 'manifest': manifest, 'config': config, 'coverage': coverage, 'sentences': _read_jsonl(root / 'sentences.jsonl'), 'occurrences': occurrences, 'groups': groups, 'pairs': pairs}


## Embedded `evaluation/representations.py`


In [ ]:
"""Storage, aggregation, and random baselines for aligned representation matrices."""
import hashlib
import json
from pathlib import Path
from typing import Any
import numpy as np

def validate_matrix(ids: list[str], matrix: np.ndarray, *, require_nonzero: bool=True) -> np.ndarray:
    values = np.asarray(matrix)
    if values.ndim != 2 or values.shape[0] != len(ids) or len(ids) != len(set(ids)):
        raise ValueError('representation IDs and matrix shape/uniqueness disagree')
    if ids != sorted(ids):
        raise ValueError('representation IDs must be stably sorted')
    if not np.isfinite(values).all():
        raise ValueError('representation matrix contains non-finite values')
    if require_nonzero and np.any(np.linalg.norm(values.astype(np.float64), axis=1) <= 1e-12):
        raise ValueError('representation matrix contains a zero-norm row')
    return values

def aggregate_occurrences(occurrence_ids: list[str], hidden: np.ndarray, groups: list[dict[str, Any]], group_kind: str) -> tuple[list[str], np.ndarray, list[int]]:
    """Mean raw hidden states over every occurrence member of each group."""
    values = validate_matrix(occurrence_ids, hidden)
    positions = {occurrence_id: index for index, occurrence_id in enumerate(occurrence_ids)}
    selected = sorted((row for row in groups if row.get('group_kind') == group_kind), key=lambda row: row['group_id'])
    group_ids, means, counts = ([], [], [])
    for group in selected:
        members = group.get('occurrence_ids')
        if not isinstance(members, list) or not members or len(members) != len(set(members)):
            raise ValueError(f"invalid membership for group {group.get('group_id')}")
        try:
            indices = [positions[member] for member in members]
        except KeyError as error:
            raise ValueError(f'group references missing occurrence {error.args[0]}') from error
        group_ids.append(group['group_id'])
        means.append(values[indices].astype(np.float64).mean(axis=0))
        counts.append(len(indices))
    matrix = np.asarray(means, dtype=np.float32)
    validate_matrix(group_ids, matrix)
    return (group_ids, matrix, counts)

def random_occurrence_baseline(occurrence_ids: list[str], *, dimension: int, seed: int) -> np.ndarray:
    if dimension <= 0 or isinstance(dimension, bool):
        raise ValueError('dimension must be a positive integer')
    if occurrence_ids != sorted(occurrence_ids) or len(occurrence_ids) != len(set(occurrence_ids)):
        raise ValueError('occurrence IDs must be unique and sorted')
    matrix = np.random.default_rng(seed).standard_normal((len(occurrence_ids), dimension)).astype(np.float32)
    return validate_matrix(occurrence_ids, matrix)

def save_representations(output_dir: str | Path, *, occurrence_ids: list[str], hidden: np.ndarray, groups: list[dict[str, Any]], provenance: dict[str, Any]) -> Path:
    target = Path(output_dir).resolve()
    if target.exists():
        raise FileExistsError(f'representation output already exists: {target}')
    validate_matrix(occurrence_ids, hidden)
    aggregates = {kind: aggregate_occurrences(occurrence_ids, hidden, groups, kind) for kind in ('character_pinyin', 'syllable')}
    target.mkdir(parents=True)
    (target / 'occurrence_ids.json').write_text(json.dumps(occurrence_ids, ensure_ascii=False) + '\n', encoding='utf-8')
    np.save(target / 'occurrence_hidden.npy', np.asarray(hidden, dtype=np.float32), allow_pickle=False)
    for kind, (ids, matrix, counts) in aggregates.items():
        (target / f'{kind}_ids.json').write_text(json.dumps(ids, ensure_ascii=False) + '\n', encoding='utf-8')
        (target / f'{kind}_counts.json').write_text(json.dumps(counts) + '\n', encoding='utf-8')
        np.save(target / f'{kind}_hidden.npy', matrix, allow_pickle=False)
    artifacts = {}
    for path in target.iterdir():
        if path.is_file():
            artifacts[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest = {'schema_version': 1, 'provenance': provenance, 'artifacts': artifacts}
    (target / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    return target


## Embedded `evaluation/encoders.py`


In [ ]:
"""Strict adapters that align encoder outputs to model-independent occurrences."""
import hashlib
import json
from pathlib import Path
import numpy as np
import torch

def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def _checkpoint_hash_contract(source_hashes: dict, experiment_manifest: dict) -> dict:
    """Complete evaluation's five-file source pin with its validated acoustic artifact pin."""
    required = {'manifest.json', 'config.json', 'vocab.json', 'samples.jsonl', 'splits.json'}
    if not isinstance(source_hashes, dict) or set(source_hashes) != required:
        raise ValueError('evaluation source experiment hash set is invalid')
    acoustic_hash = experiment_manifest.get('artifacts', {}).get('acoustic_targets.npy')
    if not isinstance(acoustic_hash, str) or len(acoustic_hash) != 64 or any((c not in '0123456789abcdef' for c in acoustic_hash)):
        raise ValueError('validated experiment manifest lacks acoustic target hash')
    return {**source_hashes, 'acoustic_targets.npy': acoustic_hash}

def extract_bilstm(bundle_path: str | Path, *, experiment_path: str | Path, checkpoint_path: str | Path, output_dir: str | Path, device: str='cpu', batch_size: int=32) -> Path:
    """Extract full-sentence BiLSTM outputs and save exact occurrence alignment."""
    if batch_size <= 0 or isinstance(batch_size, bool):
        raise ValueError('batch_size must be a positive integer')
    bundle = load_evaluation_bundle(bundle_path)
    experiment_root, checkpoint_file = (Path(experiment_path).resolve(), Path(checkpoint_path).resolve())
    source_hashes = bundle['manifest']['source_experiment']['hashes']
    for name in ('manifest.json', 'config.json', 'vocab.json', 'samples.jsonl', 'splits.json'):
        path = experiment_root / name
        if not path.is_file() or _sha256(path) != source_hashes.get(name):
            raise ValueError(f'experiment does not match evaluation source hash: {name}')
    experiment = load_experiment(experiment_root)
    expected_checkpoint_hashes = _checkpoint_hash_contract(source_hashes, experiment['manifest'])
    config, vocab = (experiment['config'], experiment['vocab'])
    checkpoint = torch.load(checkpoint_file, map_location='cpu', weights_only=True)
    model_id = canonical_model_id(checkpoint.get('model_id'))
    _, lambda_cosine = parse_model_id(model_id)
    if 'lambda_cosine' in checkpoint and checkpoint['lambda_cosine'] != lambda_cosine:
        raise ValueError('checkpoint lambda disagrees with model_id')
    if checkpoint.get('experiment_hashes') != expected_checkpoint_hashes:
        raise ValueError('checkpoint provenance does not match evaluation experiment')
    if checkpoint.get('config_sha256') != source_hashes['config.json'] or checkpoint.get('vocab_sha256') != source_hashes['vocab.json']:
        raise ValueError('checkpoint config or vocabulary provenance mismatch')
    model = build_encoder(config, vocab, model_id=model_id, seed=checkpoint['seed'])
    model.load_state_dict(checkpoint['state_dict'], strict=True)
    model.to(torch.device(device)).eval()
    samples = experiment['samples']
    occurrence_by_sample: dict[str, list[dict]] = {}
    for occurrence in bundle['occurrences']:
        occurrence_by_sample.setdefault(occurrence['sample_id'], []).append(occurrence)
    sentence_ids = [row['sample_id'] for row in bundle['sentences']]
    if set(sentence_ids) != set(occurrence_by_sample) or len(sentence_ids) != len(set(sentence_ids)):
        raise ValueError('bundle sentences and occurrence samples disagree')
    extracted: dict[str, np.ndarray] = {}
    with torch.no_grad():
        for start in range(0, len(sentence_ids), batch_size):
            batch_ids = sentence_ids[start:start + batch_size]
            selected = [samples[sample_id] for sample_id in batch_ids]
            batch = {key: value.to(device) if torch.is_tensor(value) else value for key, value in collate_experiment(selected).items()}
            outputs, output_mask = model.encode(batch)
            outputs = outputs.detach().cpu().numpy()
            if not torch.equal(output_mask, batch['char_mask']):
                raise ValueError('encoder output mask differs from source positions')
            for row_index, (sample_id, sample) in enumerate(zip(batch_ids, selected)):
                expected_positions = sorted(occurrence_by_sample[sample_id], key=lambda row: row['character_position'])
                if [row['character_position'] for row in expected_positions] != list(range(len(sample['text']))) or any((row['character'] != sample['text'][row['character_position']] for row in expected_positions)):
                    raise ValueError(f'bundle occurrences do not exactly cover source sentence {sample_id}')
                for occurrence in expected_positions:
                    extracted[occurrence['occurrence_id']] = outputs[row_index, occurrence['character_position']]
    occurrence_ids = sorted((row['occurrence_id'] for row in bundle['occurrences']))
    if set(extracted) != set(occurrence_ids):
        raise ValueError('encoder extraction did not produce every expected occurrence')
    hidden = np.stack([extracted[occurrence_id] for occurrence_id in occurrence_ids]).astype(np.float32)
    validate_matrix(occurrence_ids, hidden)
    provenance = {'adapter': 'phonetics.evaluation.encoders.extract_bilstm', 'device': str(device), 'checkpoint_path': str(checkpoint_file), 'checkpoint_sha256': _sha256(checkpoint_file), 'bundle_manifest_sha256': _sha256(Path(bundle_path).resolve() / 'manifest.json'), 'experiment_hashes': source_hashes, 'model_id': checkpoint['model_id'], 'hidden_dimension': int(hidden.shape[1])}
    return save_representations(output_dir, occurrence_ids=occurrence_ids, hidden=hidden, groups=bundle['groups'], provenance=provenance)


## Embedded `evaluation/analysis.py`


In [ ]:
"""RSA-style cosine analysis and strict paired comparison of encoder runs."""
import hashlib
import json
import math
from pathlib import Path
from typing import Any
import numpy as np
LEVELS = ('contextual', 'character_pinyin', 'syllable')
BIN_EDGES = tuple(range(0, 121, 10))

def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def distance_bin(distance: float | None) -> tuple[str, str]:
    """Return ``(status, label)`` with an explicit unbounded overflow bin."""
    if distance is None or not isinstance(distance, (int, float)) or (not math.isfinite(float(distance))):
        return ('invalid_nonfinite', 'invalid')
    value = float(distance)
    if value < 0:
        return ('invalid_negative', 'invalid')
    if value >= 120:
        return ('ok', '[120,+inf)')
    lower = int(value // 10) * 10
    return ('ok', f'[{lower},{lower + 10})')

def _rankdata(values: np.ndarray) -> np.ndarray:
    order = np.argsort(values, kind='mergesort')
    ranks = np.empty(len(values), dtype=np.float64)
    position = 0
    while position < len(values):
        end = position + 1
        while end < len(values) and values[order[end]] == values[order[position]]:
            end += 1
        ranks[order[position:end]] = (position + end - 1) / 2.0 + 1.0
        position = end
    return ranks

def spearman_result(distances: list[float], dissimilarities: list[float]) -> dict[str, Any]:
    if len(distances) != len(dissimilarities):
        raise ValueError('Spearman inputs have different lengths')
    count = len(distances)
    if count < 2:
        return {'rho': None, 'valid_pair_count': count, 'undefined_reason': 'fewer_than_two_pairs'}
    left, right = (np.asarray(distances, dtype=np.float64), np.asarray(dissimilarities, dtype=np.float64))
    if not np.isfinite(left).all() or not np.isfinite(right).all():
        raise ValueError('Spearman inputs must be finite after validity filtering')
    left_rank, right_rank = (_rankdata(left), _rankdata(right))
    if np.ptp(left_rank) == 0 or np.ptp(right_rank) == 0:
        reason = 'constant_distance' if np.ptp(left_rank) == 0 else 'constant_hidden_dissimilarity'
        return {'rho': None, 'valid_pair_count': count, 'undefined_reason': reason}
    rho = float(np.corrcoef(left_rank, right_rank)[0, 1])
    return {'rho': rho, 'valid_pair_count': count, 'undefined_reason': None}

def cosine_rows(pair_rows: list[dict], object_ids: list[str], matrix: np.ndarray) -> list[dict]:
    values = validate_matrix(object_ids, matrix)
    index = {object_id: position for position, object_id in enumerate(object_ids)}
    result = []
    for pair in pair_rows:
        try:
            left, right = (values[index[pair['left_id']]], values[index[pair['right_id']]])
        except KeyError as error:
            raise ValueError(f'pair references missing representation {error.args[0]}') from error
        cosine = float(np.dot(left.astype(np.float64), right.astype(np.float64)) / (np.linalg.norm(left.astype(np.float64)) * np.linalg.norm(right.astype(np.float64))))
        cosine = min(1.0, max(-1.0, cosine))
        bin_status, label = distance_bin(pair.get('dimsim_distance'))
        metric_status = pair.get('distance_status', 'missing_status')
        analysis_status = 'ok' if metric_status == 'ok' and bin_status == 'ok' else metric_status if metric_status != 'ok' else bin_status
        result.append({**pair, 'cosine_similarity': cosine, 'hidden_dissimilarity': 1.0 - cosine, 'analysis_status': analysis_status, 'distance_bin': label})
    return result

def summarize_rows(rows: list[dict]) -> dict[str, Any]:
    valid = [row for row in rows if row['analysis_status'] == 'ok']

    def correlation(selected: list[dict]) -> dict[str, Any]:
        return spearman_result([float(row['dimsim_distance']) for row in selected], [float(row['hidden_dissimilarity']) for row in selected])
    strata = {mask: correlation([row for row in valid if row.get('overlap_mask') == mask]) for mask in (f'{value:03b}' for value in range(8))}
    strata['no_zero_initial'] = correlation([row for row in valid if row.get('no_zero_initial_diagnostic') is True])
    bins = []
    labels = [f'[{lower},{lower + 10})' for lower in range(0, 120, 10)] + ['[120,+inf)']
    for label in labels:
        selected = [row for row in valid if row['distance_bin'] == label]
        distances = [float(row['dimsim_distance']) for row in selected]
        cosines = [float(row['cosine_similarity']) for row in selected]
        bins.append({'bin': label, 'count': len(selected), 'distance_min': min(distances) if distances else None, 'distance_max': max(distances) if distances else None, 'cosine_mean': float(np.mean(cosines)) if cosines else None})
    status_counts: dict[str, int] = {}
    for row in rows:
        status_counts[row['analysis_status']] = status_counts.get(row['analysis_status'], 0) + 1
    return {'pair_count': len(rows), 'status_counts': dict(sorted(status_counts.items())), 'overall': correlation(valid), 'strata': strata, 'bins': bins, 'uncertainty': {'status': 'not_estimated_single_seed', 'interpretation': 'fixed-pair descriptive statistic; no population or training-run confidence interval', 'iid_pair_bootstrap': False, 'p_values_reported': False}}

def _load_representation_level(root: Path, level: str) -> tuple[list[str], np.ndarray]:
    prefix = 'occurrence' if level == 'contextual' else level
    ids = json.loads((root / f'{prefix}_ids.json').read_text(encoding='utf-8'))
    matrix = np.load(root / f'{prefix}_hidden.npy', allow_pickle=False)
    return (ids, matrix)

def _plot_bins(summary: dict, path: Path, title: str) -> None:
    import matplotlib.pyplot as plt
    populated = [row for row in summary['bins'] if row['count']]
    figure, axis = plt.subplots(figsize=(10, 4))
    axis.plot([row['bin'] for row in populated], [row['cosine_mean'] for row in populated], marker='o')
    for index, row in enumerate(populated):
        axis.annotate(f"n={row['count']}", (index, row['cosine_mean']), xytext=(0, 6), textcoords='offset points', ha='center', fontsize=7)
    axis.set(title=title, xlabel='DIMSIM distance bin', ylabel='Mean raw cosine')
    axis.tick_params(axis='x', rotation=45)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)

def analyze_representations(bundle_path: str | Path, representations_dir: str | Path, *, output_dir: str | Path, run_label: str) -> Path:
    bundle, representation_root = (load_evaluation_bundle(bundle_path), Path(representations_dir).resolve())
    representation_manifest = json.loads((representation_root / 'manifest.json').read_text(encoding='utf-8'))
    required_representation_artifacts = {'occurrence_ids.json', 'occurrence_hidden.npy'}
    for level in ('character_pinyin', 'syllable'):
        required_representation_artifacts.update({f'{level}_ids.json', f'{level}_counts.json', f'{level}_hidden.npy'})
    representation_artifacts = representation_manifest.get('artifacts', {})
    if not required_representation_artifacts.issubset(representation_artifacts):
        raise ValueError('representation manifest is missing required artifacts')
    for name, digest in representation_artifacts.items():
        artifact = (representation_root / name).resolve()
        if representation_root not in artifact.parents or not artifact.is_file() or _sha256(artifact) != digest:
            raise ValueError(f'representation artifact checksum mismatch: {name}')
    bundle_manifest_sha = _sha256(Path(bundle_path).resolve() / 'manifest.json')
    if representation_manifest.get('provenance', {}).get('bundle_manifest_sha256') != bundle_manifest_sha:
        raise ValueError('representations were not extracted from this evaluation bundle')
    target = Path(output_dir).resolve()
    if target.exists():
        raise FileExistsError(f'analysis output already exists: {target}')
    target.mkdir(parents=True)
    (target / 'similarities').mkdir()
    (target / 'plots').mkdir()
    metrics, table_hashes = ({}, {})
    for level in LEVELS:
        ids, matrix = _load_representation_level(representation_root, level)
        if level == 'contextual':
            expected_ids = sorted((row['occurrence_id'] for row in bundle['occurrences']))
        else:
            expected_groups = sorted((row for row in bundle['groups'] if row['group_kind'] == level), key=lambda row: row['group_id'])
            expected_ids = [row['group_id'] for row in expected_groups]
            counts = json.loads((representation_root / f'{level}_counts.json').read_text(encoding='utf-8'))
            if counts != [row['occurrence_count'] for row in expected_groups]:
                raise ValueError(f'representation group counts disagree with bundle: {level}')
        if ids != expected_ids:
            raise ValueError(f'representation IDs disagree with complete bundle objects: {level}')
        rows = cosine_rows(bundle['pairs'][level], ids, matrix)
        table = target / 'similarities' / f'{level}.jsonl'
        table.write_text(''.join((json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in rows)), encoding='utf-8')
        table_hashes[level] = _sha256(table)
        metrics[level] = summarize_rows(rows)
        _plot_bins(metrics[level], target / 'plots' / f'{level}-bin-means.png', f'{run_label}: {level}')
    (target / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    pair_hashes = {level: bundle['manifest']['artifacts'][f'pairs/{level}.jsonl'] for level in LEVELS}
    manifest = {'schema_version': 1, 'run_label': run_label, 'bundle_manifest_sha256': bundle_manifest_sha, 'pair_plan_sha256': pair_hashes, 'representation_manifest_sha256': _sha256(representation_root / 'manifest.json'), 'similarity_table_sha256': table_hashes, 'metrics_sha256': _sha256(target / 'metrics.json')}
    (target / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    return target

def compare_paired_runs(left_dir: str | Path, right_dir: str | Path, *, output_dir: str | Path) -> Path:
    left_root, right_root = (Path(left_dir).resolve(), Path(right_dir).resolve())
    manifests = [json.loads((root / 'manifest.json').read_text(encoding='utf-8')) for root in (left_root, right_root)]
    for root, manifest in zip((left_root, right_root), manifests):
        if _sha256(root / 'metrics.json') != manifest.get('metrics_sha256'):
            raise ValueError(f'analysis metrics checksum mismatch: {root}')
        for level in LEVELS:
            if _sha256(root / 'similarities' / f'{level}.jsonl') != manifest.get('similarity_table_sha256', {}).get(level):
                raise ValueError(f'analysis similarity checksum mismatch: {root} {level}')
    for field in ('bundle_manifest_sha256', 'pair_plan_sha256'):
        if manifests[0].get(field) != manifests[1].get(field):
            raise ValueError(f'paired runs disagree on {field}')
    metrics = [json.loads((root / 'metrics.json').read_text(encoding='utf-8')) for root in (left_root, right_root)]
    paired_tables = {}
    model_fields = {'cosine_similarity', 'hidden_dissimilarity'}
    for level in LEVELS:
        tables = []
        for root in (left_root, right_root):
            tables.append([json.loads(line) for line in (root / 'similarities' / f'{level}.jsonl').read_text(encoding='utf-8').splitlines()])
        if [row['pair_id'] for row in tables[0]] != [row['pair_id'] for row in tables[1]]:
            raise ValueError(f'paired runs have different ordered pair IDs for {level}')
        if any(({key: value for key, value in left.items() if key not in model_fields} != {key: value for key, value in right.items() if key not in model_fields} for left, right in zip(*tables))):
            raise ValueError(f'paired runs disagree on common pair metadata for {level}')
        paired_tables[level] = tables
    target = Path(output_dir).resolve()
    if target.exists():
        raise FileExistsError(f'comparison output already exists: {target}')
    target.mkdir(parents=True)
    (target / 'joined').mkdir()
    result = {'left_run': manifests[0]['run_label'], 'right_run': manifests[1]['run_label'], 'levels': {}}
    joined_hashes = {}
    for level in LEVELS:
        tables = paired_tables[level]
        joined = [{**{key: value for key, value in left.items() if key not in model_fields}, 'left_cosine_similarity': left['cosine_similarity'], 'right_cosine_similarity': right['cosine_similarity'], 'left_hidden_dissimilarity': left['hidden_dissimilarity'], 'right_hidden_dissimilarity': right['hidden_dissimilarity']} for left, right in zip(*tables)]
        joined_path = target / 'joined' / f'{level}.jsonl'
        joined_path.write_text(''.join((json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in joined)), encoding='utf-8')
        joined_hashes[level] = _sha256(joined_path)
        left_rho, right_rho = (metrics[0][level]['overall']['rho'], metrics[1][level]['overall']['rho'])
        result['levels'][level] = {'left_rho': left_rho, 'right_rho': right_rho, 'delta_rho_right_minus_left': None if left_rho is None or right_rho is None else right_rho - left_rho, 'pair_count': len(tables[0]), 'strata': {}}
        for stratum in [f'{value:03b}' for value in range(8)] + ['no_zero_initial']:
            left_value = metrics[0][level]['strata'][stratum]['rho']
            right_value = metrics[1][level]['strata'][stratum]['rho']
            result['levels'][level]['strata'][stratum] = {'left_rho': left_value, 'right_rho': right_value, 'delta_rho_right_minus_left': None if left_value is None or right_value is None else right_value - left_value}
    result['uncertainty'] = {'status': 'not_estimated', 'paired_point_difference_only': True, 'iid_pair_bootstrap': False}
    output = target / 'comparison.json'
    output.write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    comparison_manifest = {'schema_version': 1, 'input_analysis_manifest_sha256': {'left': _sha256(left_root / 'manifest.json'), 'right': _sha256(right_root / 'manifest.json')}, 'comparison_sha256': _sha256(output), 'joined_sha256': joined_hashes}
    (target / 'manifest.json').write_text(json.dumps(comparison_manifest, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    return target


## Embedded `evaluation/runner.py`


In [ ]:
"""End-to-end orchestration for one immutable geometry-evaluation run."""
import hashlib
import json
import platform
import shutil
from importlib import metadata
from pathlib import Path
from typing import Any
import numpy as np

def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def _version(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def run_evaluation(bundle_path: str | Path, *, output_root: str | Path, run_id: str, adapter: str, code_provenance: dict[str, str], experiment_path: str | Path | None=None, checkpoint_path: str | Path | None=None, device: str='cpu', batch_size: int=32, random_seed: int=42, random_dimension: int | None=None) -> Path:
    """Extract or generate representations, then run all three analyses."""
    if Path(run_id).name != run_id or run_id in {'', '.', '..'}:
        raise ValueError('run_id must be one safe path component')
    if adapter not in {'bilstm', 'random'}:
        raise ValueError("adapter must be 'bilstm' or 'random'")
    if not isinstance(code_provenance, dict) or not code_provenance or any((not isinstance(key, str) or not isinstance(value, str) or len(value) != 64 or any((character not in '0123456789abcdef' for character in value)) for key, value in code_provenance.items())):
        raise ValueError('explicit non-empty code_provenance hashes are required')
    bundle_root = Path(bundle_path).resolve()
    bundle = load_evaluation_bundle(bundle_root)
    target = (Path(output_root).resolve() / run_id).resolve()
    if target.parent != Path(output_root).resolve() or target.exists():
        raise FileExistsError(f'unsafe or existing evaluation run: {target}')
    if target == bundle_root or bundle_root in target.parents:
        raise ValueError('evaluation run output must not be inside its input bundle')
    if experiment_path is not None:
        experiment_root = Path(experiment_path).resolve()
        if target == experiment_root or experiment_root in target.parents:
            raise ValueError('evaluation run output must not be inside its source experiment')
    target.mkdir(parents=True)
    try:
        representations = target / 'representations'
        if adapter == 'bilstm':
            if experiment_path is None or checkpoint_path is None:
                raise ValueError('bilstm adapter requires experiment_path and checkpoint_path')
            extract_bilstm(bundle_root, experiment_path=experiment_path, checkpoint_path=checkpoint_path, output_dir=representations, device=device, batch_size=batch_size)
            adapter_config = {'experiment_path': str(Path(experiment_path).resolve()), 'checkpoint_path': str(Path(checkpoint_path).resolve()), 'device': device, 'batch_size': batch_size}
        else:
            if random_dimension is None or random_dimension <= 0 or isinstance(random_dimension, bool):
                raise ValueError('random adapter requires a positive random_dimension')
            occurrence_ids = sorted((row['occurrence_id'] for row in bundle['occurrences']))
            hidden = random_occurrence_baseline(occurrence_ids, dimension=random_dimension, seed=random_seed)
            save_representations(representations, occurrence_ids=occurrence_ids, hidden=hidden, groups=bundle['groups'], provenance={'adapter': 'iid_standard_normal', 'seed': random_seed, 'dimension': random_dimension, 'bundle_manifest_sha256': _sha256(bundle_root / 'manifest.json')})
            adapter_config = {'random_seed': random_seed, 'random_dimension': random_dimension}
        analysis = analyze_representations(bundle_root, representations, output_dir=target / 'analysis', run_label=run_id)
        actual = {'schema_version': 1, 'run_id': run_id, 'adapter': adapter, 'adapter_config': adapter_config, 'bundle_path': str(bundle_root), 'bundle_manifest_sha256': _sha256(bundle_root / 'manifest.json'), 'representations_manifest_sha256': _sha256(representations / 'manifest.json'), 'analysis_manifest_sha256': _sha256(analysis / 'manifest.json'), 'code_provenance': code_provenance, 'environment': {'python': platform.python_version(), 'numpy': np.__version__, 'torch': _version('torch'), 'dimsim': _version('dimsim'), 'pypinyin': _version('pypinyin')}}
        (target / 'actual_config.json').write_text(json.dumps(actual, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    except Exception:
        shutil.rmtree(target)
        raise
    return target


In [ ]:
import hashlib
from pathlib import Path

EXPERIMENT_ID = "duyu-chinese-lessons-zh-cn-v1"
EXPERIMENT_MANIFEST_SHA256 = "719fc321a121c06a25c5247533564c15ce02f26e6bc82aa20c33fc567a227185"
SEED = 0
COMPONENTS_RUN_ID = "REPLACE_WITH_COMPONENTS_RUN_ID"
COMBINED_RUN_ID = "REPLACE_WITH_COMBINED_RUN_ID"
LAMBDA_COSINE = 1.0

# Kaggle examples; edit these explicit paths for attached datasets.
BUNDLE_PATH = Path("/kaggle/input/mandarin-evaluation-1-0-0/1.0.0")
OUTPUT_ROOT = Path("/kaggle/working/evaluation-runs")
EXPERIMENT_PATH = Path("/kaggle/input/mandarin-experiment-1-0-0") / EXPERIMENT_ID
RUNS_ROOT = Path("/kaggle/input/mandarin-training-runs/runs") / EXPERIMENT_ID

if hashlib.sha256((EXPERIMENT_PATH / "manifest.json").read_bytes()).hexdigest() != EXPERIMENT_MANIFEST_SHA256:
    raise ValueError("Experiment does not match the confirmed training input")
RUNS = [
    {"run_id": "components", "adapter": "bilstm", "experiment_path": EXPERIMENT_PATH, "checkpoint_path": RUNS_ROOT / "components" / COMPONENTS_RUN_ID / f"seed-{SEED}" / "best.pt"},
    {"run_id": combined_model_id(LAMBDA_COSINE), "adapter": "bilstm", "experiment_path": EXPERIMENT_PATH, "checkpoint_path": RUNS_ROOT / combined_model_id(LAMBDA_COSINE) / COMBINED_RUN_ID / f"seed-{SEED}" / "best.pt"},
    {"run_id": "random-seed-42", "adapter": "random", "random_seed": 42, "random_dimension": 1024},
]
DEVICE = "cpu"
BATCH_SIZE = 32
PAIRED_COMPARISONS = [("components", combined_model_id(LAMBDA_COSINE))]


In [ ]:
import torch
print("torch", torch.__version__, "cuda_available", torch.cuda.is_available())
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(f"Selected device {DEVICE!r} requires CUDA, but CUDA is unavailable")


In [ ]:
missing_checkpoints = [str(specification["checkpoint_path"]) for specification in RUNS
                       if specification["adapter"] != "random" and not Path(specification["checkpoint_path"]).is_file()]
if missing_checkpoints:
    raise FileNotFoundError("Training checkpoints are required before evaluation: " + ", ".join(missing_checkpoints))

completed = {}
for specification in RUNS:
    arguments = dict(specification)
    run_id = arguments.pop("run_id")
    adapter = arguments.pop("adapter")
    completed[run_id] = run_evaluation(
        BUNDLE_PATH, output_root=OUTPUT_ROOT, run_id=run_id, adapter=adapter,
        device=DEVICE, batch_size=BATCH_SIZE, code_provenance=BUILD_METADATA["source_sha256"], **arguments,
    )

for left, right in PAIRED_COMPARISONS:
    compare_paired_runs(
        completed[left] / "analysis", completed[right] / "analysis",
        output_dir=OUTPUT_ROOT / f"compare-{left}-vs-{right}",
    )

for run_id, path in completed.items():
    metrics = json.loads((path / "analysis" / "metrics.json").read_text(encoding="utf-8"))
    print(run_id, {level: values["overall"] for level, values in metrics.items()})
